In [ ]:
# pip install pandas

import pandas as pd

# Büyük veri dosyasını parça parça oku
chunks = pd.read_csv("final_combined_data.csv", chunksize=5000, low_memory=False)

filtered_data = []

for chunk in chunks:
    # Temizleme
    chunk = chunk.replace({'NA': None, 'Unknown': None, 'Not Reported': None})
    chunk['TMB'] = pd.to_numeric(chunk['TMB'], errors='coerce')
    chunk['PD-L1'] = pd.to_numeric(chunk['Clinically reported PD-L1 score'], errors='coerce')

    # Klinik eşiğe göre filtrele (TMB ≥ 3, PD-L1 ≥ 1)
    filtered = chunk[(chunk['TMB'] >= 3) & (chunk['PD-L1'] >= 1)]

    # Gerekli sütunlar eksikse at
    filtered = filtered.dropna(subset=['MSI Type', 'Durable clinical response', 'TMB', 'PD-L1'])

    # Uygun olan satırları listeye ekle
    filtered_data.append(filtered)

# Hepsini birleştir
final_df = pd.concat(filtered_data, ignore_index=True)

# Yeni düzenlenmiş veri setini kaydet
final_df.to_csv("final_filtered_clinical_dataset.csv", index=False)

print("Yeni dosya başarıyla kaydedildi: final_filtered_clinical_dataset.csv")


In [ ]:
# pip install pandas numpy matplotlib scikit-learn shap scikit-fuzzy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
import shap
import skfuzzy as fuzz
df = pd.read_csv("data/final_filtered_clinical_dataset.csv")

features = ['Clinically reported PD-L1 score', 'TMB', 'MSI Type']
target = 'Durable clinical response'

df = df.replace({'NA': np.nan, 'Unknown': np.nan, 'Not Reported': np.nan})
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df = df.dropna(subset=features + [target])

le = LabelEncoder()
df['MSI Type'] = le.fit_transform(df['MSI Type'])
df[target] = le.fit_transform(df[target])
X = df[features]
y = df[target]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("\n🔍 Classification Report:\n")
print(classification_report(y_test, y_pred))
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test[:100], check_additivity=False)

# Genel bar grafiği
if isinstance(shap_values.values, list) or shap_values.values.ndim == 3:
    values = shap_values.values[:, :, 1]
else:
    values = shap_values.values

mean_abs_shap = np.abs(values).mean(axis=0)
feature_names = X.columns.tolist()
top_indices = np.argsort(mean_abs_shap)[-10:][::-1]
top_features = [feature_names[i] for i in top_indices]
top_values = mean_abs_shap[top_indices]

plt.figure(figsize=(8, 5))
plt.barh(top_features[::-1], top_values[::-1])
plt.xlabel("Ortalama |SHAP| Değeri")
plt.title("SHAP Özellik Önem Grafiği (Genel)")
plt.tight_layout()
plt.show()
hasta_indeksi = 1 # ← buradan hastayı seçebilirsin
print(f"\n🎯 Hasta indeksi: {hasta_indeksi}")
shap_values.feature_names = X.columns.tolist()
shap.plots.waterfall(shap_values[hasta_indeksi, :, 1])
pdl1 = df.iloc[hasta_indeksi]['Clinically reported PD-L1 score']
tmb = df.iloc[hasta_indeksi]['TMB']

x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)

pdl1_low = fuzz.trapmf(x_pdl1, [0, 0, 10, 30])
pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_low = fuzz.trapmf(x_tmb, [0, 0, 5, 10])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

pdl1_l = fuzz.interp_membership(x_pdl1, pdl1_low, pdl1)
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_l = fuzz.interp_membership(x_tmb, tmb_low, tmb)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)

if tmb_h > 0.5:
    if pdl1_h > 0.3:
        suggestion = "🟢 Monoterapi (PD-1 inhibitörü) önerilir."
    else:
        suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir."
else:
    suggestion = "🔴 Daha fazla biyobelirteç verisi gerekli."

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}")
print("Öneri:", suggestion)


In [ ]:
# pip install scikit-fuzzy
import pandas as pd
import numpy as np
import skfuzzy as fuzz

# Veri setini yükle
df = pd.read_csv("final_combined_data.csv")
df = df.replace({'NA': None, 'Unknown': None, 'Not Reported': None})
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df['PD-L1'] = pd.to_numeric(df['Clinically reported PD-L1 score'], errors='coerce')
df = df.dropna(subset=['TMB', 'PD-L1'])

# Bulanık kümeleri tanımla
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

print("\n🟡 Kombinasyon önerisi verilen hastaların indeksleri:")

for i in range(len(df)):
    pdl1 = df.iloc[i]['Clinically reported PD-L1 score']
    tmb = df.iloc[i]['TMB']

    pdl1_l = fuzz.interp_membership(x_pdl1, pdl1_low, pdl1)
    pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
    tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)

    if tmb_h > 0.5 and pdl1_h <= 0.3:
        print(f"→ hasta_indeksi = {i} | PD-L1: {pdl1}, TMB: {tmb}")



In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, accuracy_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# 📄 VERİ YÜKLEME ve TEMİZLEME
df = pd.read_csv("final_filtered_clinical_dataset.csv")
df['Durable clinical response'] = df['Durable clinical response'].replace({
    "Durable clinical benefit": 1,
    "No durable clinical benefit": 0
})
df = df[df['Durable clinical response'].isin([0, 1])].copy()

# 📌 ÖZELLİKLER
features = [
    'Clinically reported PD-L1 score', 'PD-L1', 'TMB', 'MSI Type',
    'Tumor Purity', 'Mutation Count', 'Fraction Genome Altered', 'Cancer Type Detailed'
]
target = 'Durable clinical response'

# 🔧 DÖNÜŞÜMLER
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df = df.dropna(subset=features + [target])
df['MSI Type'] = LabelEncoder().fit_transform(df['MSI Type'])
df[target] = df[target].astype(int)

X = df[features]
y = df[target]

# 🔨 ÖN İŞLEME
num_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer()), ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("num", num_pipe, num_features),
                                  ("cat", cat_pipe, cat_features)])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🧪 TRAIN/TEST BÖLÜNMESİ
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# ⚖️ SADECE EĞİTİM VERİSİNE SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# 🔧 scale_pos_weight
neg = sum(y_train_resampled == 0)
pos = sum(y_train_resampled == 1)
scale_pos_weight = neg / pos

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                      scale_pos_weight=scale_pos_weight, random_state=42)
model.fit(X_train_resampled, y_train_resampled)
y_proba = model.predict_proba(X_test)[:, 1]
proba_corrected = 1 - y_proba  # class_1 yerine class_0 çıktığı için tersliyoruz
y_pred = model.predict(X_test)

# 🔍 EŞİK = 0.5 DEĞERLENDİRME
print("\n🔍 Classification Report (Threshold = 0.5):\n")
print(classification_report(y_test, y_pred))
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("📈 ROC AUC Score (terslenmiş):", roc_auc_score(y_test, proba_corrected))

# 🎯 OPTİMUM EŞİK BULMA
prec, recall, thresholds = precision_recall_curve(y_test, proba_corrected)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
print(f"\n📍 Optimum Threshold: {best_thresh:.2f}")
print(f"📈 En yüksek F1-score: {np.max(f1_scores):.3f}")

# 🧪 YENİ EŞİK İLE TAHMİN
y_pred_opt = (proba_corrected > best_thresh).astype(int)

# 🔍 OPTİMUM EŞİK DEĞERLENDİRME
print("\n🔍 Classification Report (Optimum Eşik ile):\n")
print(classification_report(y_test, y_pred_opt))
print("✅ Accuracy (optimum eşik):", accuracy_score(y_test, y_pred_opt))
print("📈 ROC AUC Score (doğru):", roc_auc_score(y_test, proba_corrected))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt)
plt.title("Optimum Eşik ile Confusion Matrix")
plt.show()

# 🧠 SHAP
explainer = shap.Explainer(model, X_train_resampled)
X_test_df = pd.DataFrame(X_test[:50], columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)
if isinstance(shap_values.values, list) or shap_values.values.ndim == 3:
    shap_values.values = shap_values.values[:, :, 1]

shap.plots.bar(shap_values)
shap.plots.waterfall(shap_values[0])

# 🤖 BULANIK MANTIK TEDAVİ ÖNERİSİ
hasta_indeksi = 6
pdl1 = df.iloc[hasta_indeksi]['Clinically reported PD-L1 score']
tmb = df.iloc[hasta_indeksi]['TMB']

x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)

if tmb_h > 0.5:
    if pdl1_h > 0.3:
        suggestion = "🟢 Monoterapi (PD-1 inhibitörü) önerilir."
    else:
        suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir."
else:
    suggestion = "🔴 Daha fazla biyobelirteç verisi gerekli."

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}")
print("Öneri:", suggestion)
print(classification_report(y_test, y_pred_opt))



In [ ]:
import numpy as np
import pandas as pd
import skfuzzy as fuzz

# PD-L1 ve TMB bulanık kümeleri
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

# Kombinasyon önerisi verilen hastaların indekslerini topla
kombinasyon_indices = []

for i in df.index:
    try:
        pdl1 = df.loc[i]['Clinically reported PD-L1 score']
        tmb = df.loc[i]['TMB']
        if pd.notnull(pdl1) and pd.notnull(tmb):
            pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
            tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
            if tmb_h > 0.5 and pdl1_h <= 0.3:
                kombinasyon_indices.append(i)
    except:
        continue

print("🔹 Kombinasyon tedavisi önerilen hasta indeksleri:")
print(kombinasyon_indices)


In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, accuracy_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# 📄 VERİ YÜKLEME ve TEMİZLEME
df = pd.read_csv("final_filtered_clinical_dataset.csv")
df['Durable clinical response'] = df['Durable clinical response'].replace({
    "Durable clinical benefit": 1,
    "No durable clinical benefit": 0
})
df = df[df['Durable clinical response'].isin([0, 1])].copy()
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df = df.dropna(subset=[
    'Clinically reported PD-L1 score', 'PD-L1', 'TMB', 'MSI Type',
    'Tumor Purity', 'Mutation Count', 'Fraction Genome Altered',
    'Cancer Type Detailed', 'Durable clinical response'
])
df['MSI Type'] = LabelEncoder().fit_transform(df['MSI Type'])
df[target] = df['Durable clinical response'].astype(int)

# 📌 ÖZELLİKLER
features = [
    'Clinically reported PD-L1 score', 'PD-L1', 'TMB', 'MSI Type',
    'Tumor Purity', 'Mutation Count', 'Fraction Genome Altered', 'Cancer Type Detailed'
]
target = 'Durable clinical response'

X = df[features]
y = df[target]

# 🔨 ÖN İŞLEME
num_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer()), ("scaler", StandardScaler())])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features)
])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🧪 TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# ⚖️ scale_pos_weight
neg = sum(y_train_resampled == 0)
pos = sum(y_train_resampled == 1)
scale_pos_weight = neg / pos

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                      scale_pos_weight=scale_pos_weight, random_state=42)
model.fit(X_train_resampled, y_train_resampled)
y_proba = model.predict_proba(X_test)[:, 1]
proba_corrected = 1 - y_proba  # çünkü class_0 çıkıyor
y_pred = model.predict(X_test)

# 🔍 STANDART EŞİK ANALİZİ
print("\n🔍 Classification Report (Threshold = 0.5):\n")
print(classification_report(y_test, y_pred))
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("📈 ROC AUC Score (terslenmiş):", roc_auc_score(y_test, proba_corrected))

# 🎯 EŞİK OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, proba_corrected)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
print(f"\n📍 Optimum Threshold: {best_thresh:.2f}")
print(f"📈 En yüksek F1-score: {np.max(f1_scores):.3f}")

# 🔁 Optimum eşikle yeniden sınıflandır
y_pred_opt = (proba_corrected > best_thresh).astype(int)
print("\n🔍 Classification Report (Optimum Eşik ile):\n")
print(classification_report(y_test, y_pred_opt))
print("✅ Accuracy (optimum eşik):", accuracy_score(y_test, y_pred_opt))
print("📈 ROC AUC Score (doğru):", roc_auc_score(y_test, proba_corrected))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt)
plt.title("Optimum Eşik ile Confusion Matrix")
plt.show()

# 🧠 SHAP
explainer = shap.Explainer(model, X_train_resampled)
X_test_df = pd.DataFrame(X_test[:50], columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)
if isinstance(shap_values.values, list) or shap_values.values.ndim == 3:
    shap_values.values = shap_values.values[:, :, 1]

shap.plots.bar(shap_values)
shap.plots.waterfall(shap_values[0])
shap_values_df = pd.DataFrame(shap_values.values, columns=feature_names)
shap_values_df.mean().sort_values(ascending=False).head(10)


# 🤖 BULANIK MANTIK TEDAVİ ÖNERİSİ
hasta_indeksi = 6  # örnek hasta
pdl1 = df.iloc[hasta_indeksi]['Clinically reported PD-L1 score']
tmb = df.iloc[hasta_indeksi]['TMB']

x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)

if tmb_h > 0.5:
    if pdl1_h > 0.3:
        suggestion = "🟢 Monoterapi (PD-1 inhibitörü) önerilir."
    else:
        suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir."
else:
    suggestion = "🔴 Daha fazla biyobelirteç verisi gerekli."

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}")
print("Öneri:", suggestion)


In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from scipy.sparse import issparse

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, accuracy_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# 📄 VERİ YÜKLEME ve TEMİZLEME
df = pd.read_csv("final_filtered_clinical_dataset.csv")
df['Durable clinical response'] = df['Durable clinical response'].replace({
    "Durable clinical benefit": 1,
    "No durable clinical benefit": 0
})
df = df[df['Durable clinical response'].isin([0, 1])].copy()
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df = df.dropna(subset=[
    'Clinically reported PD-L1 score', 'PD-L1', 'TMB', 'MSI Type',
    'Tumor Purity', 'Mutation Count', 'Fraction Genome Altered',
    'Cancer Type Detailed', 'Durable clinical response'
])
df['MSI Type'] = LabelEncoder().fit_transform(df['MSI Type'])
df['Durable clinical response'] = df['Durable clinical response'].astype(int)

# 📌 ÖZELLİKLER
features = [
    'Clinically reported PD-L1 score', 'PD-L1', 'TMB', 'MSI Type',
    'Tumor Purity', 'Mutation Count', 'Fraction Genome Altered', 'Cancer Type Detailed'
]
target = 'Durable clinical response'

X = df[features]
y = df[target]

# 🔨 ÖN İŞLEME
num_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer()), ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features)
])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🧪 TRAIN/TEST
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# ⚖️ scale_pos_weight
neg = sum(y_train_resampled == 0)
pos = sum(y_train_resampled == 1)
scale_pos_weight = neg / pos

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                      scale_pos_weight=scale_pos_weight, random_state=42)
model.fit(X_train_resampled, y_train_resampled)
y_proba = model.predict_proba(X_test)[:, 1]
proba_corrected = 1 - y_proba  # class_1 başarılı
y_pred = model.predict(X_test)

# 📈 STANDART DEĞERLENDİRME
print("\n🔍 Classification Report (Threshold = 0.5):\n")
print(classification_report(y_test, y_pred))
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("📈 ROC AUC Score (terslenmiş):", roc_auc_score(y_test, proba_corrected))

# 🎯 EŞİK OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, proba_corrected)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
print(f"\n📍 Optimum Threshold: {best_thresh:.2f}")
print(f"📈 En yüksek F1-score: {np.max(f1_scores):.3f}")

y_pred_opt = (proba_corrected > best_thresh).astype(int)
print("\n🔍 Classification Report (Optimum Eşik ile):\n")
print(classification_report(y_test, y_pred_opt))
print("✅ Accuracy (optimum eşik):", accuracy_score(y_test, y_pred_opt))
print("📈 ROC AUC Score (doğru):", roc_auc_score(y_test, proba_corrected))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt)
plt.title("Optimum Eşik ile Confusion Matrix")
plt.show()

# 🧠 SHAP
explainer = shap.Explainer(model, X_train_resampled)
X_test_df = pd.DataFrame(X_test[:50], columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)
if isinstance(shap_values.values, list) or shap_values.values.ndim == 3:
    shap_values.values = shap_values.values[:, :, 1]
shap.plots.bar(shap_values)
shap.plots.waterfall(shap_values[0])

# ✳️ TEDAVİ BAŞARI OLASILIĞI
hasta_indeksi = 6
orijinal_index = df.index[hasta_indeksi]
hasta_verisi_raw = X.iloc[hasta_indeksi:hasta_indeksi+1]
hasta_transformed = preprocessor.transform(hasta_verisi_raw)
if issparse(hasta_transformed):
    hasta_transformed = hasta_transformed.toarray()

tedavi_basari_olasiligi = model.predict_proba(hasta_transformed)[0][1]
print("\n🧪 Tahmin Edilen Tedavi Başarı Olasılığı:")
print(f"Bu hasta için tedavinin başarılı olma olasılığı: %{tedavi_basari_olasiligi*100:.2f}")

# 📌 GELİŞMİŞ TEDAVİ ÖNERİ SİSTEMİ
hasta = df.iloc[hasta_indeksi]
pdl1 = hasta.get('Clinically reported PD-L1 score', 0)
tmb = hasta.get('TMB', 0)
cancer = str(hasta.get('Cancer Type Detailed', 'unknown')).lower()
msi = str(hasta.get('MSI Type', 'unknown')).lower()
ecog = hasta.get('ECOG', 99)
metastasis = str(hasta.get('Metastatic Site', 'none')).lower()
age = hasta.get('Age', 0)
egfr_mut = hasta.get('EGFR driver', False)
braf_mut = hasta.get('BRAF driver', False)

if msi in ['msi-high', 'msi-h']:
    suggestion = "🟢 Monoterapi (PD-1 inhibitörü) önerilir (MSI-H hasta)."
elif egfr_mut:
    suggestion = "🔴 EGFR mutasyonu pozitif — PD-1 tedavisi sınırlı etkili olabilir."
elif tmb >= 20:
    suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir (yüksek TMB)."
elif tmb >= 10 and pdl1 < 1 and any(x in cancer for x in ['lung', 'melanoma', 'colon']) and ecog <= 1:
    suggestion = "🟡 Kombinasyon önerilir (orta TMB, düşük PD-L1, uygun kanser, iyi performans)."
elif pdl1 >= 50 and ecog <= 1:
    suggestion = "🟢 Monoterapi yeterlidir (yüksek PD-L1, iyi ECOG)."
elif age > 75 or ecog >= 2:
    suggestion = "🔴 Kombinasyon tedavi yan etki riski yüksek — monoterapi veya alternatif tedavi düşünülmeli."
elif metastasis != "none":
    suggestion = "🟡 Metastatik hasta — kombine rejim göz önüne alınabilir."
else:
    suggestion = "🔴 Daha fazla biyobelirteç ve klinik değerlendirme gereklidir."

print("\n📌 Klinik Temelli Gelişmiş Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}, MSI: {msi}, ECOG: {ecog}, Metastaz: {metastasis}, Age: {age}")
print("Öneri:", suggestion)


In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt
from scipy.sparse import issparse

# 📄 VERİYİ YÜKLE
df = pd.read_csv("merged_final_clinical_dataset.csv")

# 🎯 HEDEF VE ÖZELLİKLER
target = "Durable clinical response"
X = df.drop(columns=[target])
y = df[target]

# 🔧 SÜTUNLARI AYIR
numeric_features = ['TMB', 'Clinically reported PD-L1 score']
categorical_features = ['MSI Normalized', 'Age Group', 'Cancer Group']
binary_features = ['Metastasis Present']

# 📦 ÖN İŞLEME
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')  # binary alanlar doğrudan geçer

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_processed, y)

# 🔀 TRAIN-TEST
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42)

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# 🔍 TAHMİN
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# 📈 METRİKLER
print("\n📊 Classification Report:\n")
print(classification_report(y_test, y_pred))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
plt.title("Confusion Matrix")
plt.show()

# 🧠 SHAP ANALİZİ
explainer = shap.Explainer(model, X_train)
X_test_df = pd.DataFrame(X_test[:50].toarray() if issparse(X_test[:50]) else X_test[:50], columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)

# 📊 ÖZELLİK ÖNEMİ GRAFİĞİ
shap.plots.bar(shap_values)

# 🔬 TEKİL HASTA ANALİZİ (örnek: 0. hasta)
instance_index = 0
shap.plots.waterfall(shap_values[instance_index])

# 🧪 BAŞARI OLASILIK TAHMİNİ
print("\n📌 Örnek Hasta Başarı Olasılığı:")
sample = X_test_df.iloc[[instance_index]]
sample_proba = model.predict_proba(sample)[0][1]
print(f"Bu hasta için tedavinin başarılı olma olasılığı: %{sample_proba * 100:.2f}")

# 🧠 KLİNİK TEDAVİ ÖNERİSİ
# Orijinal değerleri yakalayalım
raw_patient_index = df.index[instance_index]
raw_patient = df.loc[raw_patient_index]

# Klinik parametreler
pdl1 = raw_patient['Clinically reported PD-L1 score']
tmb = raw_patient['TMB']
age_group = raw_patient['Age Group']
msi = str(raw_patient['MSI Normalized']).lower()
cancer_group = str(raw_patient['Cancer Group']).lower()
metastasis = raw_patient['Metastasis Present']

# Klinik öneri kuralları
if "msi-h" in msi:
    suggestion = "🟢 Monoterapi (PD-1 inhibitörü) önerilir (MSI-H hasta)"
elif tmb >= 20:
    suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir (yüksek TMB)"
elif tmb >= 10 and pdl1 < 1:
    suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir (orta TMB ve düşük PD-L1)"
elif pdl1 >= 50:
    suggestion = "🟢 Monoterapi önerilir (yüksek PD-L1)"
elif age_group == "Elderly" or metastasis == 1:
    suggestion = "🟡 Kombinasyon tedavi düşünülebilir ancak yaş ve/veya metastaz durumu nedeniyle düşük yan etki riski açısından monoterapi tercih edilebilir"
else:
    suggestion = "🔴 Daha fazla biyobelirteç verisi gerekli"

# Sonucu yazdır
print("\n📌 Klinik Kurallara Dayalı Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}, MSI: {msi}, Age group: {age_group}, Metastaz: {metastasis}")
print("Öneri:", suggestion)


In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, accuracy_score, roc_auc_score,
                             precision_recall_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from scipy.sparse import issparse

# 📄 VERİ YÜKLEME
df = pd.read_csv("synthetic_imbalanced_300_clinical_dataset.csv")
df = df.dropna(subset=['response', 'pd_l1', 'tmb', 'age', 'msi', 'metastasis'])

X = df[['pd_l1', 'tmb', 'age', 'msi', 'metastasis']]
y = df["response"]

# 🔧 ÖZELLİK ÖN İŞLEME
numeric_features = ['tmb', 'pd_l1']
categorical_features = ['age', 'msi']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 📤 TRAIN/TEST SPLIT
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42
)

# ⚖️ SMOTE (sadece eğitim verisine)
smote = SMOTE(random_state=42, k_neighbors=1)
X_train, y_train = smote.fit_resample(X_train_raw, y_train_raw)

# 🌲 MODEL EĞİTİMİ
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# 🔮 TAHMİN
y_proba = model.predict_proba(X_test_raw)[:, 1]

# 🎯 OPTİMUM EŞİK HESABI
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]

# 🔍 SONUÇLAR
y_pred = (y_proba > best_thresh).astype(int)

print("\n📍 Optimum Eşik Değeri:", round(best_thresh, 2))
print("\n📊 Classification Report (Optimum Eşik):\n")
print(classification_report(y_test, y_pred))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("📌 Confusion Matrix (Optimum Threshold)")
plt.show()

# 🧠 SHAP ANALİZİ
explainer = shap.Explainer(model, X_train)
X_test_df = pd.DataFrame(X_test_raw[:50].toarray() if issparse(X_test_raw[:50]) else X_test_raw[:50],
                         columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)

shap.plots.bar(shap_values)
shap.plots.waterfall(shap_values[0])

# 🤖 BULANIK MANTIK TEDAVİ ÖNERİSİ
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)

# Üyelik fonksiyonları
pdl1_high = fuzz.trapmf(x_pdl1, [30, 60, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 15, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])

# 🎯 ÖRNEK HASTA ANALİZİ
hasta_indeksi = 72
pdl1 = df.iloc[hasta_indeksi]['pd_l1']
tmb = df.iloc[hasta_indeksi]['tmb']
age_cat = df.iloc[hasta_indeksi]['age']
msi = df.iloc[hasta_indeksi]['msi']
metastasis = df.iloc[hasta_indeksi]['metastasis']

# Yaş dönüştür
age_map = {"Young": 30, "Middle-Aged": 50, "Senior": 70, "Elderly": 80}
age = age_map.get(age_cat, 60)

pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)

# 🩺 BULANIK KURAL BAZLI TEDAVİ
if "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif tmb_h > 0.5 and pdl1_h < 0.4:
    suggestion = "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük)"
elif pdl1_h > 0.6 and tmb_h > 0.6:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yüksek)"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – başka biyobelirteçler gerekli"
elif age_o > 0.6 or metastasis == 1:
    suggestion = "🟡 Kombinasyon düşünülebilir, yan etki riski varsa monoterapi tercih edilebilir"
else:
    suggestion = "🔴 Klinik değerlendirme ile karar verilmeli"

# 🧠 MODEL TAHMİNİ
sample_X = X_processed[hasta_indeksi]
proba = model.predict_proba([sample_X])[0][1]

print("\n🧠 Model Tahmini:")
print(f"Bu hasta için tedavi başarılı olma olasılığı: %{proba*100:.2f}")

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}, Yaş: {age_cat}, MSI: {msi}, Metastaz: {metastasis}")
print("Öneri:", suggestion)


In [ ]:
# 📦 GEREKLİ KÜtÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, accuracy_score, roc_auc_score,
                             precision_recall_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from scipy.sparse import issparse

# 📄 VERİ YÜKLEME
file_path = "synthetic_imbalanced_300_clinical_dataset_enriched.csv"
df = pd.read_csv(file_path)
df = df.dropna()

# 🎯 ÖZELLİKLER
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']

X = df[features]
y = df[target]

# 🔧 ÖN İŞLEME
numeric_features = ['tmb', 'pd_l1', 'ecog_score', 'treatment_start_days']
categorical_features = ['age', 'msi', 'smoking_status', 'cancer_stage']
binary_features = ['metastasis']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_processed, y)

# 🔀 TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42)

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 THRESHOLD OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

print(f"\n🔍 Optimum Eşik Değeri: {best_thresh:.2f}")
print("\n📊 Classification Report (Optimum Eşik):\n")
print(classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 🔳 CONFUSION MATRIX
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt,
                                        display_labels=["Başarısız", "Başarılı"],
                                        cmap="Blues")
plt.title(f"Optimum Eşik ile Confusion Matrix\n(Eşik = {best_thresh:.2f})")
plt.show()

# 🧠 SHAP ANALİZİ
explainer = shap.Explainer(model, X_train)
X_test_df = pd.DataFrame(X_test[:50].toarray() if issparse(X_test[:50]) else X_test[:50],
                         columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)
shap.plots.bar(shap_values)
shap.plots.waterfall(shap_values[0])

# 🧐 BULANIK MANTIK
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🌟 ÖRNEK HASTA
idx = 192
row = df.iloc[idx]

pdl1 = row['pd_l1']
tmb = row['tmb']
age = 80 if row['age'] == "Elderly" else 60
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# 🔄 BULANIK KARAR KURALLARI
if "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif tmb_h > 0.6 and pdl1_h < 0.4:
    suggestion = "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük)"
elif pdl1_h > 0.6 and tmb_h > 0.6:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yüksek)"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – başka biyobelirteçler gerekli"
elif age_o > 0.6 or metastasis == 1 or ecog_b > 0.5 or delay_l > 0.5:
    if smoking in ['current smoker', 'former smoker'] or stage in ['III', 'IV']:
        suggestion = "🟡 Kombinasyon (yüksek risk grubu – ileri evre/sigara geçmişi)"
    else:
        suggestion = "🟡 Kombinasyon düşünülebilir, yan etki riski varsa monoterapi"
else:
    suggestion = "🔴 Klinik değerlendirme ile karar verilmeli"

# 🔬 MODEL TAHMİNİ (TEK HASTA)
sample_X = X_processed[idx]
proba = model.predict_proba([sample_X])[0][1]

print("\n Model Tahmini:")
print(f"Bu hasta için tedavi başarılı olma olasılığı: %{proba * 100:.2f}")
print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}, ECOG: {ecog}, Gecikme: {delay}, Yaş (grup): {row['age']}, Sigara: {row['smoking_status']}, Evre: {stage}")
print("Öneri:", suggestion)


In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, accuracy_score, roc_auc_score,
                             precision_recall_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# 📄 VERİ YÜKLEME
df = pd.read_csv("synthetic_imbalanced_300_clinical_dataset_enriched.csv").dropna()

# 🎯 ÖZELLİKLER
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
X = df[features]
y = df[target]

# 🔧 ÖN İŞLEME
numeric_features = ['tmb', 'pd_l1', 'ecog_score', 'treatment_start_days']
categorical_features = ['age', 'msi', 'smoking_status', 'cancer_stage']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_processed, y)

# 🔀 TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42)

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 THRESHOLD OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

print(f"\n🔍 Optimum Eşik Değeri: {best_thresh:.2f}")
print("\n📊 Classification Report (Optimum Eşik):\n")
print(classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 🔳 CONFUSION MATRIX
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt,
                                        display_labels=["Başarısız", "Başarılı"],
                                        cmap="Blues")
plt.title(f"Optimum Eşik ile Confusion Matrix\n(Eşik = {best_thresh:.2f})")
plt.show()

# 👤 İNCELENECEK HASTA
idx = 192  # burayı istediğin hasta ID ile değiştir
row = df.iloc[idx]

# 🔬 MODEL TAHMİNİ
sample_X = X_processed[idx].reshape(1, -1)
proba = model.predict_proba(sample_X)[0][1]


# 🧠 SHAP ANALİZİ (Waterfall)
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])  # sadece o hasta için SHAP grafiği

# 🧐 BULANIK MANTIK
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🌟 HASTA VERİSİ
pdl1 = row['pd_l1']
tmb = row['tmb']
age = 80 if row['age'] == "Elderly" else 60
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

# 📈 BULANIK ÜYELİK
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# 🔄 BULANIK KURALLAR
# 🔄 GÜNCELLENMİŞ BULANIK KURALLAR (YUMUŞAK EŞİKLERLE)
# Risk puanı hesapla
risk_score = 0
if age_o > 0.5: risk_score += 1
if ecog_b > 0.5: risk_score += 1
if delay_l > 0.5: risk_score += 1
if metastasis == 1: risk_score += 1
if smoking in ['current smoker', 'former smoker']: risk_score += 1
if stage in ['III', 'IV']: risk_score += 1

# Karar kuralları
if "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif pdl1_h > 0.6 and tmb_h > 0.4 and risk_score <= 2:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yüksek, düşük risk)"
elif pdl1_h > 0.6 and tmb_h <= 0.4 and risk_score <= 2:
    suggestion = "🟢 Monoterapi (PD-L1 yüksek, TMB düşük, düşük risk)"
elif tmb_h > 0.6 and pdl1_h < 0.4:
    suggestion = "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük)"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – başka biyobelirteçler gerekli"
elif risk_score >= 3 and (pdl1_h > 0.4 or tmb_h > 0.4):
    suggestion = "🟡 Kombinasyon (orta-yüksek riskli hasta)"
else:
    suggestion = "🔴 Klinik değerlendirme ile karar verilmeli"

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print("\n📊 Model Tahmini:")
print(f"Bu hasta için tedavi başarılı olma olasılığı: %{proba * 100:.2f}")
print(f"PD-L1: {pdl1}, TMB: {tmb}, ECOG: {ecog}, Gecikme: {delay}, Yaş: {row['age']}, Sigara: {row['smoking_status']}, Evre: {stage}")
print("Öneri:", suggestion)


In [ ]:
# 📦 Gerekli kütüphaneler
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# 🔧 Fonksiyon: Klinik öneri + fuzzy sistem
def tedavi_onerisi_analizi(row):
    pdl1 = row['pd_l1']
    tmb = row['tmb']
    age = 80 if row['age'] == "Elderly" else 60
    msi = row['msi']
    metastasis = row['metastasis']
    ecog = row['ecog_score']
    delay = row['treatment_start_days']
    smoking = row['smoking_status'].lower()
    stage = row['cancer_stage']

    x_pdl1 = np.arange(0, 101, 1)
    x_tmb = np.arange(0, 51, 1)
    x_age = np.arange(20, 101, 1)
    x_ecog = np.arange(0, 5, 1)
    x_delay = np.arange(0, 101, 1)

    pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
    tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
    age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
    ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
    delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

    pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
    tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
    age_o = fuzz.interp_membership(x_age, age_old, age)
    ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
    delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

    risk_score = 0
    if age_o > 0.5: risk_score += 1
    if ecog_b > 0.5: risk_score += 1
    if delay_l > 0.5: risk_score += 1
    if metastasis == 1: risk_score += 1
    if smoking in ['current smoker', 'former smoker']: risk_score += 1
    if stage in ['III', 'IV']: risk_score += 1

    if "msi-h" in msi.lower():
        return "🟢 Monoterapi (MSI-H hasta)"
    elif pdl1_h > 0.6 and tmb_h > 0.4 and risk_score <= 2:
        return "🟢 Monoterapi (PD-L1 ve TMB yüksek, düşük risk)"
    elif pdl1_h > 0.6 and tmb_h <= 0.4 and risk_score <= 2:
        return "🟢 Monoterapi (PD-L1 yüksek, TMB düşük, düşük risk)"
    elif tmb_h > 0.6 and pdl1_h < 0.4:
        return "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük)"
    elif pdl1_h < 0.3 and tmb_h < 0.3:
        return "🔴 Yetersiz veri – başka biyobelirteçler gerekli"
    elif risk_score >= 3 and (pdl1_h > 0.4 or tmb_h > 0.4):
        return "🟡 Kombinasyon (orta-yüksek riskli hasta)"
    else:
        return "🔴 Klinik değerlendirme ile karar verilmeli"

# 📄 Veri yükle
df = pd.read_csv("synthetic_imbalanced_300_clinical_dataset_enriched.csv").dropna()

target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']

X = df[features]
y = df[target]

# 🔧 Ön işleme
numeric_features = ['tmb', 'pd_l1', 'ecog_score', 'treatment_start_days']
categorical_features = ['age', 'msi', 'smoking_status', 'cancer_stage']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# ⚖️ Dengeleme ve modelleme
smote = SMOTE(random_state=42, k_neighbors=1)
X_res, y_res = smote.fit_resample(X_processed, y)

X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, stratify=y_res, random_state=42)
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# 👤 Hasta ID'si (örnek: 192)
idx = 192
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)
proba = model.predict_proba(sample_X)[0][1]

# 🧠 SHAP analiz (senin makinede çalışır)
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])  # Hasta için etki analizi

# ✅ Çıktı
print("\n🔬 Klinik Temelli Tedavi Önerisi:")
print(tedavi_onerisi_analizi(row))

print("\n📊 Model Tahmini (XGBoost):")
print(f"Tedavi başarılı olma olasılığı: %{proba * 100:.2f}")
print(f"Hasta ID: {idx} → PD-L1: {row['pd_l1']}, TMB: {row['tmb']}, ECOG: {row['ecog_score']}, Gecikme: {row['treatment_start_days']}")


In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import euclidean_distances
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# 🔄 FONKSİYON: Bulanık Mantık Temelli Tedavi Önerisi
def tedavi_onerisi_analizi(row):
    pdl1 = row['pd_l1']
    tmb = row['tmb']
    age = 80 if row['age'] == "Elderly" else 60
    msi = row['msi']
    metastasis = row['metastasis']
    ecog = row['ecog_score']
    delay = row['treatment_start_days']
    smoking = row['smoking_status'].lower()
    stage = row['cancer_stage']

    x_pdl1 = np.arange(0, 101, 1)
    x_tmb = np.arange(0, 51, 1)
    x_age = np.arange(20, 101, 1)
    x_ecog = np.arange(0, 5, 1)
    x_delay = np.arange(0, 101, 1)

    pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
    tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
    age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
    ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
    delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

    pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
    tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
    age_o = fuzz.interp_membership(x_age, age_old, age)
    ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
    delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

    risk_score = sum([
        age_o > 0.5,
        ecog_b > 0.5,
        delay_l > 0.5,
        metastasis == 1,
        smoking in ['current smoker', 'former smoker'],
        stage in ['III', 'IV']
    ])

    if "msi-h" in msi.lower():
        return "🟢 Monoterapi (MSI-H hasta)"
    elif pdl1_h > 0.6 and tmb_h > 0.4 and risk_score <= 2:
        return "🟢 Monoterapi (PD-L1 ve TMB yüksek, düşük risk)"
    elif pdl1_h > 0.6 and tmb_h <= 0.4 and risk_score <= 2:
        return "🟢 Monoterapi (PD-L1 yüksek, TMB düşük, düşük risk)"
    elif tmb_h > 0.6 and pdl1_h < 0.4:
        return "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük)"
    elif pdl1_h < 0.3 and tmb_h < 0.3:
        return "🔴 Yetersiz veri – başka biyobelirteçler gerekli"
    elif risk_score >= 3 and (pdl1_h > 0.4 or tmb_h > 0.4):
        return "🟡 Kombinasyon (orta-yüksek riskli hasta)"
    else:
        return "🔴 Klinik değerlendirme ile karar verilmeli"

# 📄 VERİ YÜKLE
df = pd.read_csv("synthetic_imbalanced_300_clinical_dataset_enriched.csv").dropna()

features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
target = "response"

X = df[features]
y = df[target]

# 🔧 ÖN İŞLEME
numeric_features = ['tmb', 'pd_l1', 'ecog_score', 'treatment_start_days']
categorical_features = ['age', 'msi', 'smoking_status', 'cancer_stage']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_processed, y)

# 🔀 SPLIT ve MODEL
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

# 👤 HASTA ID SEÇ
idx = 123
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)

# 🔍 BENZER HASTA KONTROLÜ
distance = euclidean_distances(sample_X, X_processed)[0]
benzerlik_esigi = 1.0
benzer_idx = np.where((distance < benzerlik_esigi) & (distance > 0))[0]
benzer_sayi = len(benzer_idx)
benzer_basari = y.iloc[benzer_idx].mean() if benzer_sayi > 0 else None

# ✅ ÇIKTILAR
print(f"\n📌 Hasta ID: {idx}")
print("\n🧠 Tedavi Önerisi:")
print(tedavi_onerisi_analizi(row))

if benzer_sayi == 0:
    print("\n⚠️ Bu hastaya klinik olarak benzer başka veri bulunamadı.")
    print("🧪 Tahmin düşük güvenilirlikte. Klinik uzman kararı gereklidir.")
else:
    prediction = model.predict_proba(sample_X)[0][1]
    print(f"\n🔍 Benzer hasta sayısı: {benzer_sayi}")
    print(f"📈 Benzer hastalarda başarı ortalaması: %{benzer_basari*100:.2f}")
    print(f"📊 Tedavi Başarı Oranı Tahmini: %{prediction*100:.2f}")

# 📉 SHAP GRAFİĞİ
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])

In [ ]:
# 📦 Gerekli kütüphaneler
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import euclidean_distances
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# 📄 Veriyi yükle
df = pd.read_csv("synthetic_imbalanced_300_clinical_dataset_enriched.csv").dropna()

# 🎯 Özellikler
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
target = "response"
X = df[features]
y = df[target]

# 🔧 Ön işleme
numeric_features = ['tmb', 'pd_l1', 'ecog_score', 'treatment_start_days']
categorical_features = ['age', 'msi', 'smoking_status', 'cancer_stage']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')

X_processed = preprocessor.fit_transform(X)

# ⚖️ SMOTE ile sınıf dengesi sağla
smote = SMOTE(random_state=42, k_neighbors=1)
X_res, y_res = smote.fit_resample(X_processed, y)

# 🌲 Modeli eğit
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, stratify=y_res, random_state=42)
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# 👤 İncelenecek Hasta ID'si
idx = 192
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)
prediction = model.predict_proba(sample_X)[0][1]

# 🧭 Klinik olarak en yakın hastayı bul (Öklidyen mesafe ile)
mesafeler = euclidean_distances(sample_X, X_processed)[0]
benzerlik_df = pd.DataFrame({"index": range(len(mesafeler)), "mesafe": mesafeler})
benzerlik_df = benzerlik_df[benzerlik_df["index"] != idx].sort_values(by="mesafe").reset_index(drop=True)

# 🔍 En yakın 5 hastaya bak
en_yakinlar = benzerlik_df.head(5)

print(f"\n📌 İncelenen Hasta ID: {idx}")
print(f"🧠 Model Tahmini (bu hasta için): %{prediction*100:.2f}")

# 🚨 Benzer veri yoksa uyarı ver
temin_uyari = False
if en_yakinlar.iloc[0]['mesafe'] > 3.5:
    print("\n⚠️ Bu hastaya ait klinik verilere benzer örnekler çok az bulunduğu için tahmin güvenilir değildir.")
    print("⚠️ Benzer vaka sayısı yetersiz olduğundan bu durum klinik uzmanın inisiyatifine bırakılmıştır.")
    temin_uyari = True
else:
    print("\n🔍 Klinik Verisi En Yakın Hastalar:")
    for i, row_ in en_yakinlar.iterrows():
        yak_idx = int(row_["index"])
        yak_veri = df.iloc[yak_idx]
        yak_pred = model.predict_proba(X_processed[yak_idx].reshape(1, -1))[0][1]
        print(f"- Hasta ID: {yak_idx} | Mesafe: {row_['mesafe']:.2f} | Tahmin: %{yak_pred*100:.2f}")
        print(f"  -> PD-L1: {yak_veri['pd_l1']}, TMB: {yak_veri['tmb']}, ECOG: {yak_veri['ecog_score']}, Gecikme: {yak_veri['treatment_start_days']}")

if not temin_uyari:
    print("\n📌 Sonuç: Tahmin güvenilir aralıkta yapılmıştır. Yukarıda listelenen benzer hasta(lar) referans alınmıştır.")

In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# 📄 VERİ YÜKLEME
df = pd.read_csv("synthetic_imbalanced_300_clinical_dataset_enriched.csv").dropna()

# 🎯 ÖZELLİKLER
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis', 'ecog_score',
            'smoking_status', 'cancer_stage', 'treatment_start_days']
X = df[features]
y = df[target]

# 🔧 ÖN İŞLEME
numeric_features = ['tmb', 'pd_l1', 'ecog_score', 'treatment_start_days']
categorical_features = ['age', 'msi', 'smoking_status', 'cancer_stage']
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🔀 VERİ BÖLME
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# 👤 İNCELENECEK HASTA
idx = 192
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)

# ✅ BENZERLİK ANALİZİ
dists = np.linalg.norm(X_train - sample_X, axis=1)
similarity_score = np.min(dists)
threshold = 7.5  # Çok uzak hasta için eşik

if similarity_score < threshold:
    proba = model.predict_proba(sample_X)[0][1]
    explainer = shap.Explainer(model, X_train)
    hasta_df = pd.DataFrame(sample_X, columns=feature_names)
    hasta_shap = explainer(hasta_df, check_additivity=False)
    shap.plots.waterfall(hasta_shap[0])
    print(f"🔹 Model Tahmini: Bu hasta için tedavi başarılı olma olasılığı: %{proba * 100:.2f}")
else:
    print("🔴 Bu hastaya benzer klinik veri eğitim setinde bulunamadı.")
    print("⚠️ Model tahmini yapılmadı. Karar hekimin klinik değerlendirmesine bırakılmalıdır.")

# 🧠 BULANIK MANTIK (her durumda çalışır)
# Üyelik fonksiyonları
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# Hasta verileri
pdl1 = row['pd_l1']
tmb = row['tmb']
age = 80 if row['age'] == "Elderly" else 60
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

# Üyelik hesaplama
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# Risk puanı ve karar kuralları
risk_score = 0
if age_o > 0.5: risk_score += 1
if ecog_b > 0.5: risk_score += 1
if delay_l > 0.5: risk_score += 1
if metastasis == 1: risk_score += 1
if smoking in ['current smoker', 'former smoker']: risk_score += 1
if stage in ['III', 'IV']: risk_score += 1

if "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif pdl1_h > 0.6 and tmb_h > 0.4 and risk_score <= 2:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yüksek, düşük risk)"
elif pdl1_h > 0.6 and tmb_h <= 0.4 and risk_score <= 2:
    suggestion = "🟢 Monoterapi (PD-L1 yüksek, TMB düşük, düşük risk)"
elif tmb_h > 0.6 and pdl1_h < 0.4:
    suggestion = "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük)"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – başka biyobelirteçler gerekli"
elif risk_score >= 3 and (pdl1_h > 0.4 or tmb_h > 0.4):
    suggestion = "🟡 Kombinasyon (orta-yüksek riskli hasta)"
else:
    suggestion = "🔴 Klinik değerlendirme ile karar verilmeli"

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print("Öneri:", suggestion)


In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, accuracy_score, roc_auc_score,
                             precision_recall_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# 📄 VERİ YÜKLEME
df = pd.read_csv("Dengeli_Klinik_Veri_Seti.csv").dropna()

# 🎯 ÖZELLİKLER
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
X = df[features]
y = df[target]

# 🔧 ÖN İŞLEME
numeric_features = ['tmb', 'pd_l1', 'ecog_score', 'treatment_start_days']
categorical_features = ['age', 'msi', 'smoking_status', 'cancer_stage']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_processed, y)

# 🔀 TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42)

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 THRESHOLD OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

print(f"\n🔍 Optimum Eşik Değeri: {best_thresh:.2f}")
print("\n📊 Classification Report (Optimum Eşik):\n")
print(classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 🔳 CONFUSION MATRIX
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt,
                                        display_labels=["Başarısız", "Başarılı"],
                                        cmap="Blues")
plt.title(f"Optimum Eşik ile Confusion Matrix\n(Eşik = {best_thresh:.2f})")
plt.show()

# 👤 İNCELENECEK HASTA
idx = 13  # burayı istediğin hasta ID ile değiştir
row = df.iloc[idx]

# 🔬 MODEL TAHMİNİ
sample_X = X_processed[idx].reshape(1, -1)
proba = model.predict_proba(sample_X)[0][1]


# 🧠 SHAP ANALİZİ (Waterfall)
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])  # sadece o hasta için SHAP grafiği

# 🧐 BULANIK MANTIK
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🌟 HASTA VERİSİ
pdl1 = row['pd_l1']
tmb = row['tmb']
age = 80 if row['age'] == "Elderly" else 60
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

# 📈 BULANIK ÜYELİK
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# 🔄 BULANIK KURALLAR
# 🔄 GÜNCELLENMİŞ BULANIK KURALLAR (YUMUŞAK EŞİKLERLE)
# Risk puanı hesapla
# 🔄 DAHA HASSAS RİSK PUANI
risk_score = 0
if age_o > 0.5: risk_score += 1
if ecog_b > 0.5: risk_score += 2  # ECOG daha ağır etki
if delay_l > 0.5: risk_score += 1
if metastasis == 1: risk_score += 1
if smoking in ['current smoker', 'former smoker']: risk_score += 1
if stage in ['III', 'IV']: risk_score += 1

# 🌟 TEDAVİ ÖNERİSİ (KLINIK HASSASIYETLE GÜNCELLENMİŞ)
if proba < 0.2:
    suggestion = "🔴 Tedavi başarısı çok düşük (%{:.2f}) – klinik değerlendirme önerilir".format(proba * 100)
elif "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif pdl1_h > 0.7 and tmb_h > 0.6 and risk_score <= 3:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yüksek, risk düşük-orta)"
elif pdl1_h > 0.7 and tmb_h <= 0.4 and risk_score <= 2:
    suggestion = "🟢 Monoterapi (PD-L1 çok yüksek, risk düşük)"
elif pdl1_h > 0.6 and tmb_h > 0.6 and risk_score > 3:
    suggestion = "🟡 Kombinasyon (PD-L1 ve TMB yüksek, ama risk de yüksek)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score <= 3:
    suggestion = "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük, risk yönetilebilir)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score > 3:
    suggestion = "🔴 Yetersiz veri – diğer biyobelirteçler gerekli"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – PD-L1 ve TMB düşük"
elif risk_score >= 4 and (pdl1_h > 0.5 or tmb_h > 0.5):
    suggestion = "🟡 Kombinasyon (yüksek risk ama PD-L1/TMB destekliyor)"
else:
    suggestion = "🔴 Klinik değerlendirme önerilir"


print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print("\n📊 Model Tahmini:")
print(f"Bu hasta için tedavi başarılı olma olasılığı: %{proba * 100:.2f}")
print(f"PD-L1: {pdl1}, TMB: {tmb}, ECOG: {ecog}, Gecikme: {delay}, Yaş: {row['age']}, Sigara: {row['smoking_status']}, Evre: {stage}")
print("Öneri:", suggestion)


In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier

# 📄 VERİ YÜKLEME
df = pd.read_csv("synthetic_imbalanced_300_clinical_dataset_enriched.csv").dropna()

# 🧹 VERİ ÖN İŞLEME
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
df['age'] = df['age'].map(age_map)
X = df[features]
y = df[target]

# 🔧 PIPELINE
numeric_features = ['tmb', 'pd_l1', 'age', 'ecog_score', 'treatment_start_days']
categorical_features = ['msi', 'smoking_status', 'cancer_stage']
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🔀 VERİYİ AYIR
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# 🌲 MODEL EĞİTİMİ
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 EŞİK OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

# 📊 SONUÇLARI YAZDIR
print(f"\n🔍 Optimum Eşik Değeri: {best_thresh:.2f}")
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 👤 İNCELENECEK HASTA (örnek: ID = 13)
idx = 1
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)
proba = model.predict_proba(sample_X)[0][1]
proba_calibrated = 1 / (1 + np.exp(-((proba - 0.5) * 6)))


# 🧠 SHAP ANALİZİ
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])

# 🧮 FUZZY GİRİŞLER
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🔢 GİRİŞ VERİLERİ
pdl1 = row['pd_l1']
tmb = row['tmb']
age = row['age']
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

# 🔍 ÜYELİKLER
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# ⚖️ RİSK HESABI
risk_score = 0
if age_o > 0.5: risk_score += 1
if ecog_b > 0.5: risk_score += 2
if delay_l > 0.5: risk_score += 1
if metastasis == 1: risk_score += 1
if smoking in ['current smoker', 'former smoker']: risk_score += 1
if stage in ['III', 'IV']: risk_score += 1

# 🧠 ENTEGRE TEDAVİ ÖNERİSİ
if proba < 0.2:
    suggestion = f"🔴 Tedavi başarısı çok düşük (%{proba*100:.2f}) – klinik değerlendirme önerilir"
elif "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif pdl1_h > 0.7 and tmb_h > 0.6 and risk_score <= 3:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yüksek, risk düşük-orta)"
elif pdl1_h > 0.7 and tmb_h <= 0.4 and risk_score <= 2:
    suggestion = "🟢 Monoterapi (PD-L1 çok yüksek, risk düşük)"
elif pdl1_h > 0.6 and tmb_h > 0.6 and risk_score > 3:
    suggestion = "🟡 Kombinasyon (PD-L1 ve TMB yüksek, ama risk yüksek)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score <= 3:
    suggestion = "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük, risk yönetilebilir)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score > 3:
    suggestion = "🔴 Yetersiz veri – diğer biyobelirteçler gerekli"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – PD-L1 ve TMB düşük"
elif risk_score >= 4 and (pdl1_h > 0.5 or tmb_h > 0.5):
    suggestion = "🟡 Kombinasyon (yüksek risk ama PD-L1/TMB destekliyor)"
else:
    suggestion = "🔴 Klinik değerlendirme önerilir"

# 📌 SONUÇ
print("\n📊 Model Tahmini:")
print(f"Bu hasta için tedavi başarılı olma olasılığı: %{proba_calibrated * 100:.2f}")
print(f"PD-L1: {pdl1}, TMB: {tmb}, ECOG: {ecog}, Gecikme: {delay}, Yaş: {age}, Sigara: {smoking}, Evre: {stage}")
print("Öneri:", suggestion)


In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier

# 📄 VERİ YÜKLEME
df = pd.read_csv("SemiFinal_DataSet.csv").dropna()

# 🧹 VERİ ÖN İŞLEME
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
df['age'] = df['age'].map(age_map)
X = df[features]
y = df[target]

# 🔧 PIPELINE
numeric_features = ['tmb', 'pd_l1', 'age', 'ecog_score', 'treatment_start_days']
categorical_features = ['msi', 'smoking_status', 'cancer_stage']
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🔀 VERİYİ AYIR
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# 🌲 MODEL EĞİTİMİ
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 EŞİK OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

# 📊 SONUÇLARI YAZDIR
print(f"\n🔍 Optimum Eşik Değeri: {best_thresh:.2f}")
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 👤 İNCELENECEK HASTA (örnek: ID = 13)
idx = 197
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)
proba = model.predict_proba(sample_X)[0][1]

# 🔄 DAHA GERÇEKÇİ PROBA DENGELEME (YUMUŞATILMIŞ TAHMİN)
def smooth_sigmoid(p):
    return 0.5 + 0.5 * np.tanh((p - 0.5) * 2)

proba_calibrated = smooth_sigmoid(proba)

# 🧮 FUZZY GİRİŞLER
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🔢 GİRİŞ VERİLERİ
pdl1 = row['pd_l1']
tmb = row['tmb']
age = row['age']
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

# 🔍 ÜYELİKLER
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# ⚖️ RİSK HESABI (geçici olarak kaldırıldı)
risk_score = 0
if age_o > 0.5: risk_score += 1
if ecog_b > 0.5: risk_score += 2
if delay_l > 0.5: risk_score += 1
if metastasis == 1: risk_score += 1
if smoking in ['current smoker', 'former smoker']: risk_score += 1
if stage in ['Stage III', 'Stage IV']: risk_score += 1

# 🧠 ENTEGRE TEDAVİ ÖNERİSİ
if proba_calibrated < 0.2:
    suggestion = f"🔴 Tedavi başarısı çok düşük (%{proba_calibrated*100:.2f}) – klinik değerlendirme önerilir"
elif "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif pdl1_h > 0.7 and tmb_h > 0.6 and risk_score <= 3:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yüksek, risk düşük-orta)"
elif pdl1_h > 0.7 and tmb_h <= 0.4 and risk_score <= 2:
    suggestion = "🟢 Monoterapi (PD-L1 çok yüksek, risk düşük)"
elif pdl1_h > 0.6 and tmb_h > 0.6 and risk_score > 3:
    suggestion = "🟡 Kombinasyon (PD-L1 ve TMB yüksek, ama risk yüksek)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score <= 3:
    suggestion = "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük, risk yönetilebilir)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score > 3:
    suggestion = "🔴 Yetersiz veri – diğer biyobelirteçler gerekli"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – PD-L1 ve TMB düşük"
elif risk_score >= 4 and (pdl1_h > 0.5 or tmb_h > 0.5):
    suggestion = "🟡 Kombinasyon (yüksek risk ama PD-L1/TMB destekliyor)"
else:
    suggestion = "🔴 Klinik değerlendirme önerilir"

# 📍 SONUÇ
print("\n📈 Model Tahmini:")
print(f"Bu hasta için tedavi başarılı olma olasılığı: %{proba_calibrated * 100:.2f}")
print(f"PD-L1: {pdl1}, TMB: {tmb}, ECOG: {ecog}, Gecikme: {delay}, Yaş: {age}, Sigara: {smoking}, Evre: {stage}")
print("Öneri:", suggestion)

In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier

# 📄 VERİ YÜKLEME
df = pd.read_csv("SemiFinal_DataSet.csv").dropna()

# 🧹 VERİ ÖN İŞLEME
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
df['age'] = df['age'].map(age_map)
X = df[features]
y = df[target]

# 🔧 PIPELINE
numeric_features = ['tmb', 'pd_l1', 'age', 'ecog_score', 'treatment_start_days']
categorical_features = ['msi', 'smoking_status', 'cancer_stage']
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🔀 VERİYİ AYIR
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# 🌲 MODEL EĞİTİMİ
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 EŞİK OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

# 📊 SONUÇLARI YAZDIR
print(f"\n🔍 Optimum Eşik Değeri: {best_thresh:.2f}")
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 👤 İNCELENECEK HASTA (örnek: ID = 13)
idx = 192
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)
proba = model.predict_proba(sample_X)[0][1]

# 🔄 DAHA GERÇEKÇİ PROBA DENGELEME (YUMUŞATILMIŞ TAHMİN)
def smooth_sigmoid(p):
    return 0.5 + 0.5 * np.tanh((p - 0.5) * 2)

proba_calibrated = smooth_sigmoid(proba)

# 🧮 FUZZY GİRİŞLER
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🔢 GİRİŞ VERİLERİ
pdl1 = row['pd_l1']
tmb = row['tmb']
age = row['age']
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

# 🔍 ÜYELİKLER
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# ⚖️ RİSK HESABI (geçici olarak kaldırıldı)
r# ⚖️ RİSK HESABI (güncellendi: evre ağırlığı artırıldı)
risk_score = 0
if age_o > 0.5: risk_score += 1
if ecog_b > 0.5: risk_score += 2
if delay_l > 0.5: risk_score += 1
if metastasis == 1: risk_score += 1
if smoking in ['current smoker', 'former smoker']: risk_score += 1
if stage in ['Stage III', 'Stage IV']: risk_score += 2  # Burada güncelleme yapıldı

# 🧠 ENTEGRE TEDAVİ ÖNERİSİ
if proba_calibrated < 0.2:
    suggestion = f"🔴 Tedavi başarısı çok düşük (%{proba_calibrated*100:.2f}) – klinik değerlendirme önerilir"
elif "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif pdl1_h > 0.7 and tmb_h > 0.6 and risk_score <= 3:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yüksek, risk düşük-orta)"
elif pdl1_h > 0.7 and tmb_h <= 0.4 and risk_score <= 2:
    suggestion = "🟢 Monoterapi (PD-L1 çok yüksek, risk düşük)"
elif pdl1_h > 0.6 and tmb_h > 0.6 and risk_score > 3:
    suggestion = "🟡 Kombinasyon (PD-L1 ve TMB yüksek, ama risk yüksek)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score <= 3:
    suggestion = "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük, risk yönetilebilir)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score > 3:
    suggestion = "🔴 Yetersiz veri – diğer biyobelirteçler gerekli"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – PD-L1 ve TMB düşük"
elif risk_score >= 4 and (pdl1_h > 0.5 or tmb_h > 0.5):
    suggestion = "🟡 Kombinasyon (yüksek risk ama PD-L1/TMB destekliyor)"
else:
    suggestion = "🔴 Klinik değerlendirme önerilir"

# 📍 SONUÇ
print("\n📈 Model Tahmini:")
print(f"Bu hasta için tedavi başarılı olma olasılığı: %{proba_calibrated * 100:.2f}")
print(f"PD-L1: {pdl1}, TMB: {tmb}, ECOG: {ecog}, Gecikme: {delay}, Yaş: {age}, Sigara: {smoking}, Evre: {stage}")
print("Öneri:", suggestion)

# 📊 SHAP GRAFİĞİ
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])

In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier

# 📄 VERİ YÜKLEME
df = pd.read_csv("SemiFinal_DataSet.csv").dropna()

# 🧹 VERİ ÖN İŞLEME
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
df['age'] = df['age'].map(age_map)
X = df[features]
y = df[target]

# 🔧 PIPELINE
numeric_features = ['tmb', 'pd_l1', 'age', 'ecog_score', 'treatment_start_days']
categorical_features = ['msi', 'smoking_status', 'cancer_stage']
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🔀 VERİYİ AYIR
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# 🌲 MODEL EĞİTİMİ
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 EŞİK OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

# 📊 SONUÇLARI YAZDIR
print(f"\n🔍 Optimum Eşik Değeri: {best_thresh:.2f}")
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 👤 İNCELENECEK HASTA
idx = 195  # buraya örnek hasta ID girilebilir
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)
proba = model.predict_proba(sample_X)[0][1]

# 🔄 PROBA YUMUŞATMA
def smooth_sigmoid(p):
    return 0.5 + 0.5 * np.tanh((p - 0.5) * 2)

proba_calibrated = smooth_sigmoid(proba)

# 🧮 FUZZY GİRİŞLER
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🔢 HASTA VERİLERİ
pdl1 = row['pd_l1']
tmb = row['tmb']
age = row['age']
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

# 🔍 ÜYELİKLER
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# ⚖️ RİSK HESABI
risk_score = 0
if age_o > 0.5: risk_score += 1
if ecog_b > 0.5: risk_score += 2
if delay_l > 0.5: risk_score += 1
if metastasis == 1: risk_score += 1
if smoking in ['current smoker', 'former smoker']: risk_score += 1
if stage in ['Stage III', 'Stage IV']: risk_score += 1

# 🧠 TEDAVİ ÖNERİSİ (KLİNİK DUYARLILIKLA)
if proba_calibrated < 0.2:
    suggestion = f"🔴 Tedavi başarısı çok düşük (%{proba_calibrated*100:.2f}) – klinik değerlendirme önerilir"
elif "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif pdl1_h > 0.7 and tmb_h > 0.6 and risk_score <= 3 and stage in ['Stage I', 'Stage II']:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yüksek, risk düşük-orta, evre erken)"
elif pdl1_h > 0.7 and tmb_h <= 0.4 and risk_score <= 2 and stage in ['Stage I', 'Stage II']:
    suggestion = "🟢 Monoterapi (PD-L1 çok yüksek, risk düşük, evre erken)"
elif pdl1_h > 0.6 and tmb_h > 0.6 and (risk_score > 3 or stage in ['Stage III', 'Stage IV']):
    suggestion = "🟡 Kombinasyon (PD-L1 ve TMB yüksek, ama risk veya evre yüksek)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score <= 3:
    suggestion = "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük, risk yönetilebilir)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score > 3:
    suggestion = "🔴 Yetersiz veri – diğer biyobelirteçler gerekli"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – PD-L1 ve TMB düşük"
elif risk_score >= 4 and (pdl1_h > 0.5 or tmb_h > 0.5):
    suggestion = "🟡 Kombinasyon (yüksek risk ama PD-L1/TMB destekliyor)"
else:
    suggestion = "🔴 Klinik değerlendirme önerilir"

# 📍 SONUÇ
print("\n📈 Model Tahmini:")
print(f"Bu hasta için tedavi başarılı olma olasılığı: %{proba_calibrated * 100:.2f}")
print(f"PD-L1: {pdl1}, TMB: {tmb}, ECOG: {ecog}, Gecikme: {delay}, Yaş: {age}, Sigara: {smoking}, Evre: {stage}")
print("Öneri:", suggestion)

# 📊 SHAP ANALİZİ
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0], max_display=10)


In [ ]:
# 📦 GEREKLI KUTUPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier

# 📄 VERI YUKLEME
df = pd.read_csv("SemiFinal_DataSet.csv").dropna()

# 🧹 VERI ON ISLEME
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
df['age'] = df['age'].map(age_map)
X = df[features]
y = df[target]

# 🔧 PIPELINE
numeric_features = ['tmb', 'pd_l1', 'age', 'ecog_score', 'treatment_start_days']
categorical_features = ['msi', 'smoking_status', 'cancer_stage']
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🔀 VERIYI AYIR
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# 🌲 MODEL EGITIMI
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 ESIK OPTIMIZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

# 📊 SONUCLARI YAZDIR
print(f"\n🔍 Optimum Esik Degeri: {best_thresh:.2f}")
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 👤 INCELENECEK HASTA (ornek: ID = 197)
idx = 5
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)
proba = model.predict_proba(sample_X)[0][1]

# 🔄 DAHA GERCEKCI PROBA DENGELEME (YUMUSATILMIS TAHMIN)
def smooth_sigmoid(p):
    return 0.5 + 0.5 * np.tanh((p - 0.5) * 2)

proba_calibrated = smooth_sigmoid(proba)

# 🔧 EK KLINIK DUZELTME: PD-L1 dusukse asiri iyimserligi baskila
if row['pd_l1'] < 15 and proba_calibrated > 0.7:
    proba_calibrated -= 0.3
    proba_calibrated = max(0, proba_calibrated)

# 🧮 FUZZY GIRISLER
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🔢 GIRIS VERILERI
pdl1 = row['pd_l1']
tmb = row['tmb']
age = row['age']
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

# 🔍 UYELIKLER
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# ⚖️ RISK HESABI
risk_score = 0
if age_o > 0.5: risk_score += 1
if ecog_b > 0.5: risk_score += 2
if delay_l > 0.5: risk_score += 1
if metastasis == 1: risk_score += 1
if smoking in ['current smoker', 'former smoker']: risk_score += 1
if stage in ['Stage III', 'Stage IV']: risk_score += 1

# 🧠 ENTEGRE TEDAVI ONERISI
if proba_calibrated < 0.2:
    suggestion = f"🔴 Tedavi basarisi cok dusuk (%{proba_calibrated*100:.2f}) – klinik degerlendirme onerilir"
elif "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif pdl1_h > 0.7 and tmb_h > 0.6 and risk_score <= 3:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yuksek, risk dusuk-orta)"
elif pdl1_h > 0.7 and tmb_h <= 0.4 and risk_score <= 2:
    suggestion = "🟢 Monoterapi (PD-L1 cok yuksek, risk dusuk)"
elif pdl1_h > 0.6 and tmb_h > 0.6 and risk_score > 3:
    suggestion = "🟡 Kombinasyon (PD-L1 ve TMB yuksek, ama risk yuksek)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score <= 3:
    suggestion = "🟡 Kombinasyon (TMB yuksek, PD-L1 dusuk, risk yonetilebilir)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score > 3:
    suggestion = "🔴 Yetersiz veri – diger biyobelirtecler gerekli"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – PD-L1 ve TMB dusuk"
elif risk_score >= 4 and (pdl1_h > 0.5 or tmb_h > 0.5):
    suggestion = "🟡 Kombinasyon (yuksek risk ama PD-L1/TMB destekliyor)"
else:
    suggestion = "🔴 Klinik degerlendirme onerilir"

# 🔽 GEREKCE
rationale = []
if pdl1_h > 0.6: rationale.append("PD-L1 yuksek")
if tmb_h > 0.6: rationale.append("TMB yuksek")
if ecog_b > 0.5: rationale.append("ECOG dusuk degil")
if delay_l > 0.5: rationale.append("Tedavi gecikmeli")
if risk_score >= 4: rationale.append("Risk yuksek")

# 📅 SONUC
print("\n📁 Model Tahmini:")
print(f"Bu hasta icin tedavi basarili olma olasiligi: %{proba_calibrated * 100:.2f}")
print(f"PD-L1: {pdl1}, TMB: {tmb}, ECOG: {ecog}, Gecikme: {delay}, Yas: {age}, Sigara: {smoking}, Evre: {stage}")
print("Oneri:", suggestion)
if rationale:
    print("Gerekce:", ", ".join(rationale))

# 🧰 SHAP ANALIZI
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])



In [ ]:
# 📦 GEREKLI KUTUPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier

# 📄 VERI YUKLEME
df = pd.read_csv("data/Lung_Immunotherapy_Risks.csv").dropna()

# 🧹 VERI ON ISLEME
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
df['age'] = df['age'].map(age_map)
X = df[features]
y = df[target]

# 🔧 PIPELINE
numeric_features = ['tmb', 'pd_l1', 'age', 'ecog_score', 'treatment_start_days']
categorical_features = ['msi', 'smoking_status', 'cancer_stage']
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🔀 VERIYI AYIR
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# 🌲 MODEL EGITIMI
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 ESIK OPTIMIZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

# 📊 SONUCLARI YAZDIR
print(f"\n🔍 Optimum Esik Degeri: {best_thresh:.2f}")
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 👤 INCELENECEK HASTA (ornek: ID = 197)
idx = 18
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)
proba = model.predict_proba(sample_X)[0][1]

# 🔄 DAHA GERCEKCI PROBA DENGELEME (YUMUSATILMIS TAHMIN)
def smooth_sigmoid(p):
    return 0.5 + 0.5 * np.tanh((p - 0.5) * 2)

proba_calibrated = smooth_sigmoid(proba)

# 🔧 EK KLINIK DUZELTME: PD-L1 dusukse asiri iyimserligi baskila
if row['pd_l1'] < 15 and proba_calibrated > 0.7:
    proba_calibrated -= 0.3
    proba_calibrated = max(0, proba_calibrated)

# 🧮 FUZZY GIRISLER
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🔢 GIRIS VERILERI
pdl1 = row['pd_l1']
tmb = row['tmb']
age = row['age']
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

# 🔍 UYELIKLER
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# ⚖️ RISK HESABI
risk_score = 0
if age_o > 0.5: risk_score += 1
if ecog_b > 0.5: risk_score += 2
if delay_l > 0.5: risk_score += 1
if metastasis == 1: risk_score += 1
if smoking in ['current smoker', 'former smoker']: risk_score += 1
if stage in ['Stage III', 'Stage IV']: risk_score += 1

# 🧠 ENTEGRE TEDAVI ONERISI
if proba_calibrated < 0.2:
    suggestion = f"🔴 Tedavi basarisi cok dusuk (%{proba_calibrated*100:.2f}) – klinik degerlendirme onerilir"
elif "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif pdl1_h > 0.7 and tmb_h > 0.6 and risk_score <= 3:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yuksek, risk dusuk-orta)"
elif pdl1_h > 0.7 and tmb_h <= 0.4 and risk_score <= 2:
    suggestion = "🟢 Monoterapi (PD-L1 cok yuksek, risk dusuk)"
elif pdl1_h > 0.6 and tmb_h > 0.6 and risk_score > 3:
    suggestion = "🟡 Kombinasyon (PD-L1 ve TMB yuksek, ama risk yuksek)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score <= 3:
    suggestion = "🟡 Kombinasyon (TMB yuksek, PD-L1 dusuk, risk yonetilebilir)"
elif tmb_h > 0.6 and pdl1_h < 0.4 and risk_score > 3:
    suggestion = "🔴 Yetersiz veri – diger biyobelirtecler gerekli"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – PD-L1 ve TMB dusuk"
elif risk_score >= 4 and (pdl1_h > 0.5 or tmb_h > 0.5):
    suggestion = "🟡 Kombinasyon (yuksek risk ama PD-L1/TMB destekliyor)"
else:
    suggestion = "🔴 Klinik degerlendirme onerilir"

# 🔽 GEREKCE
rationale = []
if pdl1_h > 0.6: rationale.append("PD-L1 yuksek")
if tmb_h > 0.6: rationale.append("TMB yuksek")
if ecog_b > 0.5: rationale.append("ECOG dusuk degil")
if delay_l > 0.5: rationale.append("Tedavi gecikmeli")
if risk_score >= 4: rationale.append("Risk yuksek")

# 📅 SONUC
print("\n📁 Model Tahmini:")
print(f"Bu hasta icin tedavi basarili olma olasiligi: %{proba_calibrated * 100:.2f}")
print(f"PD-L1: {pdl1}, TMB: {tmb}, ECOG: {ecog}, Gecikme: {delay}, Yas: {age}, Sigara: {smoking}, Evre: {stage}")
print("Oneri:", suggestion)
if rationale:
    print("Gerekce:", ", ".join(rationale))

# 🧰 SHAP ANALIZI
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])



In [ ]:
# 📦 GEREKLI KUTUPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier

# 📄 VERI YUKLEME
df = pd.read_csv("data/Lung_Immunotherapy_Risks.csv").dropna()

# 🧹 VERI ON ISLEME
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
df['age'] = df['age'].map(age_map)
X = df[features]
y = df[target]

# 🔧 PIPELINE
numeric_features = ['tmb', 'pd_l1', 'age', 'ecog_score', 'treatment_start_days']
categorical_features = ['msi', 'smoking_status', 'cancer_stage']
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🔀 VERIYI AYIR
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# 🌲 MODEL EGITIMI
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 ESIK OPTIMIZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

print(f"\n🔍 Optimum Esik Degeri: {best_thresh:.2f}")
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 👤 INCELENECEK HASTA
idx = 64
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)
proba = model.predict_proba(sample_X)[0][1]

# 🔄 PROBABILITE YUMUSATMA
def smooth_sigmoid(p):
    return 0.5 + 0.5 * np.tanh((p - 0.5) * 2)
proba_calibrated = smooth_sigmoid(proba)
if row['pd_l1'] < 15 and proba_calibrated > 0.7:
    proba_calibrated = max(0, proba_calibrated - 0.3)

# 🧮 FUZZY GIRISLER
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🔢 HASTA VERILERI
pdl1 = row['pd_l1']
tmb = row['tmb']
age = row['age']
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

# 🔍 UYELIKLER
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# 🩺 KOMPLIKASYON BAZ RISKLARI
base_risks = {
    "Fatigue": 0.30,
    "Rash": 0.15,
    "Colitis": 0.10,
    "Hepatitis": 0.07,
    "Pneumonitis": 0.06,
    "Hypothyroidism": 0.10
}

# ⚡ FUZZY RISK KATSAYISI HESAPLAMA
def complication_risk(comp):
    factor = 1.0
    if comp == "Fatigue":
        if age_o > 0.5: factor *= 1.3
        if ecog_b > 0.5: factor *= 1.4
        if metastasis == 1: factor *= 1.2
    elif comp == "Rash":
        if pdl1_h > 0.5: factor *= 1.3
        if tmb_h > 0.5: factor *= 1.2
        if "non" in smoking: factor *= 1.1
    elif comp == "Colitis":
        if tmb_h > 0.5: factor *= 1.3
        if delay_l > 0.5: factor *= 1.1
        if metastasis == 1: factor *= 1.1
    elif comp == "Hepatitis":
        if tmb_h > 0.5: factor *= 1.2
        if age_o > 0.5: factor *= 1.1
    elif comp == "Pneumonitis":
        if "smoker" in smoking: factor *= 1.5
        if ecog_b > 0.5: factor *= 1.4
        if stage in ['Stage III', 'Stage IV']: factor *= 1.2
    elif comp == "Hypothyroidism":
        if pdl1_h > 0.5: factor *= 1.2
        if tmb_h > 0.5: factor *= 1.2

    # Katsayiyi normalize et (0.5-2 arasi)
    factor = np.clip(factor, 0.5, 2.0)
    return base_risks[comp] * factor

# 🔢 TUM KOMPLIKASYON RISKLERI
complication_results = {c: complication_risk(c) for c in base_risks}

# 📊 SONUCLARI GOSTER
print("\n📁 Model Tahmini:")
print(f"Bu hasta icin tedavi basarili olma olasiligi: %{proba_calibrated * 100:.2f}")
print(f"PD-L1: {pdl1}, TMB: {tmb}, ECOG: {ecog}, Gecikme: {delay}, Yas: {age}, Sigara: {smoking}, Evre: {stage}")

print("\n💉 Komplikasyon Riskleri:")
for c, r in complication_results.items():
    print(f" - {c}: %{r*100:.1f}")

# 🧰 SHAP ANALIZI
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])

In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

# 📄 VERİ YÜKLEME
df = pd.read_csv("data/Lung_Immunotherapy_Risks.csv").dropna()

target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']

# Eğer yaş kategorik ise sayısal değere çevir
age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
if df['age'].dtype == 'object':
    df['age'] = df['age'].map(age_map)

X = df[features]
y = df[target]

# 🔧 VERİ ÖN İŞLEME
numeric_features = ['tmb', 'pd_l1', 'age', 'ecog_score', 'treatment_start_days']
categorical_features = ['msi', 'smoking_status', 'cancer_stage']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_processed = preprocessor.fit_transform(X)

# 🔀 VERİYİ AYIR
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42
)

# 🌲 MODEL + KALİBRASYON
xgb_model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

calibrated_model = CalibratedClassifierCV(
    base_estimator=xgb_model,
    method='sigmoid',  # stabil kalibrasyon
    cv=3
)
calibrated_model.fit(X_train, y_train)

# 🧰 SHAP EXPLAINER (XGB modelinden alıyoruz)
xgb_model.fit(X_train, y_train)  # SHAP için ayrı fit
explainer = shap.TreeExplainer(xgb_model)

# 🧮 FUZZY RISK VE PENALTY HESABI (SABİT KURAL)
def calculate_fuzzy_penalty(row):
    # Sabit fuzzy kurallar
    x_pdl1 = np.arange(0, 101, 1)
    x_tmb = np.arange(0, 51, 1)
    x_age = np.arange(20, 101, 1)
    x_ecog = np.arange(0, 5, 1)
    x_delay = np.arange(0, 101, 1)

    pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
    tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
    age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
    ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
    delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

    # Üyelik değerleri
    pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, row['pd_l1'])
    tmb_h = fuzz.interp_membership(x_tmb, tmb_high, row['tmb'])
    age_o = fuzz.interp_membership(x_age, age_old, row['age'])
    ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, row['ecog_score'])
    delay_l = fuzz.interp_membership(x_delay, delay_late, row['treatment_start_days'])

    # Risk skoru
    risk_score = (age_o > 0.5) + 2*(ecog_b > 0.5) + (delay_l > 0.5)
    penalty = 0.2 * (ecog_b + delay_l)  # sabit penalty

    return risk_score, penalty

# 🔎 TEK HASTA TAHMİNİ
def predict_patient(patient_id):
    row = df.iloc[patient_id]
    X_row = preprocessor.transform([row[features]])
    prob = calibrated_model.predict_proba(X_row)[0][1]  # başarı olasılığı

    # SHAP etkisi
    shap_values = explainer.shap_values(X_row)
    negative_shap_penalty = np.mean([v for v in shap_values[0] if v < 0]) * -1

    # Fuzzy risk
    risk_score, fuzzy_penalty = calculate_fuzzy_penalty(row)

    # Final olasılık
    final_prob = max(0, min(1, prob - fuzzy_penalty*0.1 - negative_shap_penalty*0.05))

    print(f"📁 Hasta ID: {patient_id}")
    print(f"Bu hasta için tedavi başarılı olma olasılığı: %{final_prob*100:.2f}")
    print(f"Risk Skoru: {risk_score} | Penalty: {fuzzy_penalty:.3f} | SHAP Negatif: {negative_shap_penalty:.3f}")

# 🔹 ÖRNEK KULLANIM
predict_patient(18)  # Hasta ID girmen yeterli

In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier

# 📄 VERİ YÜKLEME
df = pd.read_csv("data/Lung_Immunotherapy_Risks.csv").dropna()

# 🧹 VERİ ÖN İŞLEME
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']

age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
df['age'] = df['age'].map(age_map)

X = df[features]
y = df[target]

# 🔧 PIPELINE
numeric_features = ['tmb', 'pd_l1', 'age', 'ecog_score', 'treatment_start_days']
categorical_features = ['msi', 'smoking_status', 'cancer_stage']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🔀 VERİYİ AYIR
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42
)

# 🌲 MODEL EĞİTİMİ
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 EŞİK OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

print(f"\n🔍 Optimum Eşik Değeri: {best_thresh:.2f}")
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 👤 İNCELENECEK HASTA
idx = 64
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)

# 1️⃣ Ham model olasılığı
proba_model = model.predict_proba(sample_X)[0][1]
print(f"\n🔹 Ham Model Olasılığı: {proba_model:.6f}")

# 2️⃣ Olasılığı smooth sigmoid ile yumuşat
def smooth_sigmoid(p):
    return 0.5 + 0.5 * np.tanh((p - 0.5) * 2)

proba_calibrated = smooth_sigmoid(proba_model)
print(f"🔹 Sigmoid ile Yumuşatılmış Olasılık: {proba_calibrated:.6f}")

# 3️⃣ Eğer PD-L1 düşükse penalize et
if row['pd_l1'] < 15 and proba_calibrated > 0.7:
    proba_calibrated = max(0, proba_calibrated - 0.3)
print(f"🔹 PD-L1 Sonrası Olasılık: {proba_calibrated:.6f}")

# 🧮 FUZZY GİRİŞLER
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🔹 Üyelik Dereceleri
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, row['pd_l1'])
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, row['tmb'])
age_o = fuzz.interp_membership(x_age, age_old, row['age'])
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, row['ecog_score'])
delay_l = fuzz.interp_membership(x_delay, delay_late, row['treatment_start_days'])

print(f"🔹 Fuzzy Üyelikler -> PD-L1: {pdl1_h:.2f}, TMB: {tmb_h:.2f}, Age: {age_o:.2f}, ECOG: {ecog_b:.2f}, Delay: {delay_l:.2f}")

# 🔹 Fuzzy Başarı Katsayısı
success_factor = 1.0
if pdl1_h < 0.3: success_factor *= 0.85
if ecog_b > 0.7: success_factor *= 0.85
if age_o > 0.6 and row['metastasis'] == 1: success_factor *= 0.9
success_factor = np.clip(success_factor, 0.5, 2.0)

print(f"🔹 Fuzzy Başarı Katsayısı: {success_factor:.2f}")

# 🔹 Son olasılık
proba_final = proba_calibrated * success_factor
print(f"\n✅ Bu hasta icin tedavi basarili olma olasiligi: %{proba_final*100:.4f}")
print(f"PD-L1: {row['pd_l1']}, TMB: {row['tmb']}, ECOG: {row['ecog_score']}, "
      f"Gecikme: {row['treatment_start_days']}, Yas: {row['age']}, "
      f"Sigara: {row['smoking_status'].lower()}, Evre: {row['cancer_stage']}")

# 🧰 SHAP ANALİZİ
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])

In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_recall_curve
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

# 📄 VERİ YÜKLEME
df = pd.read_csv("data/Lung_Immunotherapy_Risks.csv").dropna()

# 🧹 VERİ ÖN İŞLEME
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']

# Yaş kategorilerini sayısal değerlere çevir
age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
df['age'] = df['age'].map(age_map)

X = df[features]
y = df[target]

# 🔧 PIPELINE
numeric_features = ['tmb', 'pd_l1', 'age', 'ecog_score', 'treatment_start_days']
categorical_features = ['msi', 'smoking_status', 'cancer_stage']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🔀 VERİYİ AYIR
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42
)

# 🌲 MODEL EĞİTİMİ + KALİBRASYON
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
cal_model = CalibratedClassifierCV(model, method='isotonic', cv=5)
cal_model.fit(X_train, y_train)

y_proba = cal_model.predict_proba(X_test)[:, 1]

# 🌟 EŞİK OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

print(f"\n🔍 Optimum Eşik Değeri: {best_thresh:.2f}")
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 👤 İNCELENECEK HASTA
idx = 64  # Örnek hasta indeksi
row = df.iloc[idx]
# ✅ İndeks uyumsuzluğunu önlemek için doğru satırı seçiyoruz
row_index = df.index.get_loc(idx)
sample_X = X_processed[row_index].reshape(1, -1)

# 1️⃣ Kalibre edilmiş model olasılığı
proba_model = cal_model.predict_proba(sample_X)[0][1]
print(f"\n🔹 Kalibre Edilmiş Model Olasılığı: {proba_model:.6f}")

# 🧮 FUZZY GİRİŞLER
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🔹 Üyelik Dereceleri
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, row['pd_l1'])
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, row['tmb'])
age_o = fuzz.interp_membership(x_age, age_old, row['age'])
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, row['ecog_score'])
delay_l = fuzz.interp_membership(x_delay, delay_late, row['treatment_start_days'])

print(f"🔹 Fuzzy Üyelikler -> PD-L1: {pdl1_h:.2f}, TMB: {tmb_h:.2f}, Age: {age_o:.2f}, ECOG: {ecog_b:.2f}, Delay: {delay_l:.2f}")

# 🔹 Fuzzy Başarı Katsayısı
success_factor = 1.2
if pdl1_h < 0.3: success_factor *= 0.9
if ecog_b > 0.7: success_factor *= 0.85
if age_o > 0.6 and row['metastasis'] == 1: success_factor *= 0.9
success_factor = np.clip(success_factor, 1.0, 2.0)

print(f"🔹 Fuzzy Başarı Katsayısı: {success_factor:.2f}")

# 🔹 Fuzzy ile kombine edilmiş olasılık (sigmoid harmanlama)
proba_final = 1 - (1 - proba_model) ** success_factor
proba_final = min(proba_final, 1.0)

print(f"\n✅ Bu hasta için tedavi başarılı olma olasılığı: %{proba_final*100:.2f}")
print(f"PD-L1: {row['pd_l1']}, TMB: {row['tmb']}, ECOG: {row['ecog_score']}, "
      f"Gecikme: {row['treatment_start_days']}, Yas: {row['age']}, "
      f"Sigara: {row['smoking_status'].lower()}, Evre: {row['cancer_stage']}")

# 🧰 SHAP ANALİZİ
explainer = shap.Explainer(cal_model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7) ÇIKTI: Tek hasta için kalibre olasılık + fuzzy harman + SHAP waterfall
# ─────────────────────────────────────────────────────────────────────────────
# (Bu bölümün kodu, aşağıda tanımlanan fonksiyonlar ve değişkenler hazırlandıktan sonra
#  en alttaki "if __name__ == '__main__':" bloğunda çalıştırılır.)
# ─────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────
# 6) Yardımcı fonksiyonlar: yoğun (dense) çeviri, tek hasta seçimi, güvenli yazdırma
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

def to_dense(X):
    """Sparse ise dense np.array'e çevirir."""
    return X.toarray() if hasattr(X, "toarray") else X

def select_patient_row(df, features, iloc_index):
    """
    Aynı preprocessor'ı kullanabilmek için tek hastayı features subset'iyle dataframe olarak döndürür.
    iloc_index: df.iloc[iloc_index] (0 tabanlı)
    """
    row = df.iloc[iloc_index]
    sample_df = pd.DataFrame([row[features]])
    return row, sample_df

def safe_lower(x):
    return str(x).lower() if pd.notnull(x) else "unknown"


# ─────────────────────────────────────────────────────────────────────────────
# 5) Fuzzy katsayı hesaplama (klinisyen sezgilerini modele enjekte eden katman)
# ─────────────────────────────────────────────────────────────────────────────
import skfuzzy as fuzz

def compute_fuzzy_success_factor(row):
    """
    PD-L1, TMB, yaş, ECOG, tedavi gecikmesi için basit trapez üyelikleriyle
    sezgisel bir başarı katsayısı üretir. (1.0–2.0 aralığına kırpılır)
    """
    # Evrenler
    x_pdl1  = np.arange(0, 101, 1)
    x_tmb   = np.arange(0, 51, 1)
    x_age   = np.arange(20, 101, 1)
    x_ecog  = np.arange(0, 5, 1)
    x_delay = np.arange(0, 101, 1)

    # Üyelik fonksiyonları (basit sezgisel eşikler)
    pdl1_high = fuzz.trapmf(x_pdl1,  [30, 50, 100, 100])
    tmb_high  = fuzz.trapmf(x_tmb,   [5, 10, 50, 50])
    age_old   = fuzz.trapmf(x_age,   [60, 70, 100, 100])
    ecog_bad  = fuzz.trapmf(x_ecog,  [2, 3, 4, 4])
    delay_lt  = fuzz.trapmf(x_delay, [30, 60, 100, 100])

    # Üyelik dereceleri
    pdl1_h = fuzz.interp_membership(x_pdl1,  pdl1_high, row['pd_l1'])
    tmb_h  = fuzz.interp_membership(x_tmb,   tmb_high,  row['tmb'])
    age_o  = fuzz.interp_membership(x_age,   age_old,   row['age'])
    ecog_b = fuzz.interp_membership(x_ecog,  ecog_bad,  row['ecog_score'])
    d_late = fuzz.interp_membership(x_delay, delay_lt,  row['treatment_start_days'])

    # Basit sezgisel kurallar
    success_factor = 1.2
    if pdl1_h < 0.3:
        success_factor *= 0.9
    if ecog_b > 0.7:
        success_factor *= 0.85
    if age_o > 0.6 and row.get('metastasis', 0) == 1:
        success_factor *= 0.9

    # 1.0–2.0 aralığına kırp
    success_factor = np.clip(success_factor, 1.0, 2.0)

    return {
        "success_factor": float(success_factor),
        "memberships": {
            "pdl1_high": float(pdl1_h),
            "tmb_high": float(tmb_h),
            "age_old": float(age_o),
            "ecog_bad": float(ecog_b),
            "delay_late": float(d_late),
        }
    }


# ─────────────────────────────────────────────────────────────────────────────
# 4) SHAP: Kalibre edilmiş model olasılığını açıklamak için Permutation Explainer
# ─────────────────────────────────────────────────────────────────────────────
import shap

def build_calibrated_permutation_explainer(cal_model, X_train_dense, feature_names):
    """
    CalibratedClassifierCV doğrudan açıklanamaz. Bunun yerine model fonksiyonu (predict_proba[:,1])
    ve eğitim verisini masker olarak verip permutation explainer kuruyoruz.
    """
    f = lambda X: cal_model.predict_proba(X)[:, 1]
    explainer = shap.Explainer(
        f,
        X_train_dense,
        algorithm="permutation",
        feature_names=feature_names
    )
    return explainer


# ─────────────────────────────────────────────────────────────────────────────
# 3) Model eğitimi: XGBoost + isotonic kalibrasyon, metrikler ve en iyi eşik
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_recall_curve
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

def train_and_evaluate(X_processed, y, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X_processed, y, test_size=0.2, stratify=y, random_state=random_state
    )

    # XGB + kalibrasyon
    model = XGBClassifier(
        eval_metric='logloss',
        random_state=random_state,
        n_estimators=300,
        max_depth=4,
        learning_rate=0.07,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0
    )
    cal_model = CalibratedClassifierCV(model, method='isotonic', cv=5)
    cal_model.fit(X_train, y_train)

    # Olasılıklar ve en iyi F1 eşiği (precision_recall_curve çıkışı: prec/rec = n+1, thresholds = n)
    y_proba = cal_model.predict_proba(X_test)[:, 1]
    prec, rec, thr = precision_recall_curve(y_test, y_proba)
    f1_scores = 2 * (prec[1:] * rec[1:]) / (prec[1:] + rec[1:] + 1e-6)
    best_thresh = thr[np.argmax(f1_scores)]
    y_pred_opt = (y_proba >= best_thresh).astype(int)

    report = classification_report(y_test, y_pred_opt)
    acc = round(accuracy_score(y_test, y_pred_opt), 4)
    auc = round(roc_auc_score(y_test, y_proba), 4)

    return {
        "cal_model": cal_model,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "y_proba_test": y_proba,
        "best_thresh": float(best_thresh),
        "report": report,
        "acc": acc,
        "auc": auc
    }


# ─────────────────────────────────────────────────────────────────────────────
# 2) Ön işleme hattı: sayısal/katıtsal ayır, standardize et, one-hot uygula
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

def make_preprocessor(numeric_features, categorical_features):
    numeric_pipeline = Pipeline([("scaler", StandardScaler())])
    categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ])
    return preprocessor


# ─────────────────────────────────────────────────────────────────────────────
# 1) Veri ve ayarlar: yükleme, hedef/özellikler, yaş haritalama
# ─────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

def load_and_prepare(csv_path):
    df = pd.read_csv(csv_path).dropna()

    target = "response"
    features = [
        'pd_l1', 'tmb', 'age', 'msi', 'metastasis',
        'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days'
    ]

    # Yaş kategorilerini sayısala çevir
    age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
    if df['age'].dtype == 'O':
        df['age'] = df['age'].map(age_map).fillna(df['age'])

    # Özellik seti
    X = df[features].copy()
    y = df[target].copy()

    # Preprocessor
    numeric_features = ['tmb', 'pd_l1', 'age', 'ecog_score', 'treatment_start_days']
    categorical_features = ['msi', 'smoking_status', 'cancer_stage']
    preprocessor = make_preprocessor(numeric_features, categorical_features)

    # Fit-transform
    X_processed = preprocessor.fit_transform(X)
    feature_names = preprocessor.get_feature_names_out()

    return df, X, y, preprocessor, X_processed, feature_names, features


# ─────────────────────────────────────────────────────────────────────────────
# ANA AKIŞ (7 ➜ 1 mantığıyla açıklanmış, ama çalışma sırası doğru)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 1) Veri ve ayarlar
    CSV_PATH = "data/Lung_Immunotherapy_Risks.csv"
    df, X, y, preprocessor, X_processed, feature_names, features = load_and_prepare(CSV_PATH)

    # 2) (Zaten load_and_prepare içinde kuruldu)

    # 3) Model eğit + değerlendir
    out = train_and_evaluate(X_processed, y, random_state=42)
    cal_model   = out["cal_model"]
    X_train     = out["X_train"]
    X_test      = out["X_test"]
    y_test      = out["y_test"]
    y_proba     = out["y_proba_test"]
    best_thresh = out["best_thresh"]

    print(f"\n🔍 Optimum Eşik Değeri: {best_thresh:.4f}")
    print("\n📊 Classification Report:\n", out["report"])
    print("✅ Accuracy:", out["acc"])
    print("📈 ROC AUC Score:", out["auc"])

    # Eğitim setini dense'e çevir (SHAP masker için)
    X_train_dense = to_dense(X_train)

    # 4) SHAP explainer (kalibre edilmiş olasılığı açıklamak için permutation)
    explainer = build_calibrated_permutation_explainer(
        cal_model=cal_model,
        X_train_dense=X_train_dense,
        feature_names=feature_names
    )

    # 5) Fuzzy katsayı (tek hasta)
    idx = 64  # 65. satır (0 tabanlı)
    row, sample_df = select_patient_row(df, features, iloc_index=idx)

    # Tek hastayı aynı preprocessor’dan geçir, dense yap ve isimlendir
    sample_X = preprocessor.transform(sample_df)
    sample_X_dense = to_dense(sample_X)
    hasta_df = pd.DataFrame(sample_X_dense, columns=feature_names)

    fuzzy = compute_fuzzy_success_factor(row)
    success_factor = fuzzy["success_factor"]

    # 6) Tahminler ve harman (kalibre olasılık + fuzzy)
    proba_model = cal_model.predict_proba(sample_X_dense)[0, 1]
    proba_final = 1 - (1 - proba_model) ** success_factor
    proba_final = float(min(proba_final, 1.0))

    print("\n──── Tek Hasta Özeti ────")
    print(f"🔹 Kalibre Edilmiş Model Olasılığı: {proba_model:.6f}")
    print(f"🔹 Fuzzy Üyelikler -> PD-L1:{fuzzy['memberships']['pdl1_high']:.2f}, "
          f"TMB:{fuzzy['memberships']['tmb_high']:.2f}, Yaş:{fuzzy['memberships']['age_old']:.2f}, "
          f"ECOG:{fuzzy['memberships']['ecog_bad']:.2f}, Gecikme:{fuzzy['memberships']['delay_late']:.2f}")
    print(f"🔹 Fuzzy Başarı Katsayısı: {success_factor:.2f}")
    print(f"✅ Bu hasta için tedavi başarılı olma olasılığı: %{proba_final*100:.2f}")

    print(
        "Özellikler -> "
        f"PD-L1:{row['pd_l1']}, TMB:{row['tmb']}, ECOG:{row['ecog_score']}, "
        f"Gecikme:{row['treatment_start_days']}, Yaş:{row['age']}, "
        f"Sigara:{safe_lower(row['smoking_status'])}, Evre:{row['cancer_stage']}"
    )

    # 7) SHAP waterfall (kalibre edilmiş olasılığın açıklaması)
    # Not: Jupyter/GUI dışı ortamlarda grafik penceresi açılmayabilir.
    # YENİ (doğru)
shap_values = explainer(hasta_df)   # check_additivity verme
shap.plots.waterfall(shap_values[0])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RESPONSE (başarı olasılığı) + IO_GROUP (gruplanmış tedavi tipi) TAHMİNİ
# XGB + sigmoid kalibrasyon + fuzzy + SHAP  |  Multiclass XGB (prefit calib)
# ─────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, accuracy_score, roc_auc_score,
                             precision_recall_curve, f1_score)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
import shap
import skfuzzy as fuzz

# ============================
# Yardımcılar
# ============================
def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else X

def ensure_float32(X):
    return X.astype(np.float32)

def select_patient_row(df, features, iloc_index):
    row = df.iloc[iloc_index]
    sample_df = pd.DataFrame([row[features]])
    return row, sample_df

def safe_lower(x):
    return str(x).lower() if pd.notnull(x) else "unknown"

# ============================
# Fuzzy (1.0–1.5)
# ============================
def compute_fuzzy_success_factor(row):
    x_pdl1  = np.arange(0, 101, 1)
    x_tmb   = np.arange(0, 51, 1)
    x_age   = np.arange(20, 101, 1)
    x_ecog  = np.arange(0, 5, 1)
    x_delay = np.arange(0, 121, 1)

    pdl1_high = fuzz.trapmf(x_pdl1,  [30, 50, 100, 100])
    tmb_high  = fuzz.trapmf(x_tmb,   [5, 10, 50, 50])
    age_old   = fuzz.trapmf(x_age,   [60, 70, 100, 100])
    ecog_bad  = fuzz.trapmf(x_ecog,  [2, 3, 4, 4])
    delay_lt  = fuzz.trapmf(x_delay, [30, 60, 120, 120])

    pdl1_h = fuzz.interp_membership(x_pdl1,  pdl1_high, row['pd_l1'])
    tmb_h  = fuzz.interp_membership(x_tmb,   tmb_high,  row['tmb'])
    age_o  = fuzz.interp_membership(x_age,   age_old,   row['age'])
    ecog_b = fuzz.interp_membership(x_ecog,  ecog_bad,  row['ecog_score'])
    d_late = fuzz.interp_membership(x_delay, delay_lt,  row['treatment_start_days'])

    success_factor = 1.15
    if pdl1_h < 0.3: success_factor *= 0.95
    if ecog_b > 0.7: success_factor *= 0.90
    if age_o > 0.6 and row.get('metastasis', 0) == 1: success_factor *= 0.93
    success_factor = float(np.clip(success_factor, 1.0, 1.5))

    return {
        "success_factor": success_factor,
        "memberships": {
            "pdl1_high": float(pdl1_h),
            "tmb_high": float(tmb_h),
            "age_old": float(age_o),
            "ecog_bad": float(ecog_b),
            "delay_late": float(d_late),
        }
    }

# ============================
# SHAP (response)
# ============================
def build_calibrated_permutation_explainer(cal_model, X_train_dense, feature_names):
    f = lambda X: cal_model.predict_proba(X)[:, 1]
    explainer = shap.Explainer(
        f, X_train_dense, algorithm="permutation", feature_names=feature_names
    )
    return explainer

# ============================
# RESPONSE modeli
# ============================
def train_and_evaluate_response(X_processed, y, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X_processed, y, test_size=0.2, stratify=y, random_state=random_state
    )
    X_train = ensure_float32(X_train); X_test  = ensure_float32(X_test)
    y_train = y_train.astype(np.int32); y_test = y_test.astype(np.int32)

    pos = int(y_train.sum()); neg = int(len(y_train) - pos)
    scale_pos_weight = neg / max(pos, 1)

    base = XGBClassifier(
        eval_metric='logloss',
        random_state=random_state,
        n_estimators=400,
        max_depth=4,
        learning_rate=0.06,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight
    )

    cal_model = CalibratedClassifierCV(base, method='sigmoid', cv=5)
    cal_model.fit(X_train, y_train)

    y_proba = cal_model.predict_proba(X_test)[:, 1]
    prec, rec, thr = precision_recall_curve(y_test, y_proba)
    f1_scores = 2 * (prec[1:] * rec[1:]) / (prec[1:] + rec[1:] + 1e-6)
    best_thresh = thr[np.argmax(f1_scores)]
    y_pred_opt = (y_proba >= best_thresh).astype(int)

    report = classification_report(y_test, y_pred_opt)
    acc = round(accuracy_score(y_test, y_pred_opt), 4)
    auc = round(roc_auc_score(y_test, y_proba), 4)

    return {
        "cal_model": cal_model,
        "X_train": X_train,
        "y_test": y_test,
        "y_proba_test": y_proba,
        "best_thresh": float(best_thresh),
        "report": report,
        "acc": acc,
        "auc": auc
    }

# ============================
# IO_GROUP modeli (4 sınıf) — prefit kalibrasyon
# ============================
def train_and_evaluate_io_group(X_processed, y_group_str, random_state=42):
    le = LabelEncoder()
    y_group = le.fit_transform(y_group_str)
    classes = le.classes_

    # Train/Test
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_processed, y_group, test_size=0.2, stratify=y_group, random_state=random_state
    )
    X_tr = ensure_float32(X_tr); X_te = ensure_float32(X_te)
    y_tr = y_tr.astype(np.int32); y_te = y_te.astype(np.int32)

    # Train → Train/Calib
    X_tr_in, X_cal, y_tr_in, y_cal = train_test_split(
        X_tr, y_tr, test_size=0.25, stratify=y_tr, random_state=random_state
    )
    X_tr_in = ensure_float32(X_tr_in); X_cal = ensure_float32(X_cal)
    y_tr_in = y_tr_in.astype(np.int32); y_cal = y_cal.astype(np.int32)

    # Örnek ağırlığı (1/frekans)
    vc = pd.Series(y_tr_in).value_counts()
    inv = 1.0 / vc
    sw_tr_in = pd.Series(y_tr_in).map(inv).values.astype(np.float32)

    # XGB: multi:softprob (weight sadece burada)
    base = XGBClassifier(
        eval_metric='mlogloss',
        random_state=random_state,
        n_estimators=350,
        max_depth=4,
        learning_rate=0.07,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        objective='multi:softprob',
        num_class=len(classes)
    )
    base.fit(X_tr_in, y_tr_in, sample_weight=sw_tr_in)

    # Prefit sigmoid kalibrasyon (OvR)
    cal_model = CalibratedClassifierCV(base, method='sigmoid', cv='prefit')
    cal_model.fit(X_cal, y_cal)

    # Değerlendirme
    y_proba = cal_model.predict_proba(X_te)
    y_pred  = y_proba.argmax(axis=1)
    acc = round(accuracy_score(y_te, y_pred), 4)
    f1m = round(f1_score(y_te, y_pred, average='macro'), 4)
    report = classification_report(y_te, y_pred, target_names=classes)

    return {
        "cal_model": cal_model,
        "label_encoder": le,
        "classes": classes,
        "acc": acc,
        "f1_macro": f1m,
        "report": report
    }

# ============================
# Ön işleme
# ============================
def make_preprocessor(numeric_features, categorical_features):
    numeric_pipeline = Pipeline([("scaler", StandardScaler())])
    categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ])
    return preprocessor

# ============================
# Veri yükleme + IO_TYPE → IO_GROUP
# ============================
def load_and_prepare(csv_path):
    df = pd.read_csv(csv_path).dropna(subset=[
        'pd_l1','tmb','age','msi','metastasis','ecog_score',
        'smoking_status','cancer_stage','treatment_start_days','response'
    ])

    # IO_TYPE → IO_GROUP eşlemesi (varsa)
    if 'io_type' in df.columns:
        map_to_group = {
            'PD-1 monotherapy': 'Monotherapy (PD-1/PD-L1)',
            'PD-L1 monotherapy': 'Monotherapy (PD-1/PD-L1)',
            'Chemo + PD-1': 'Chemo + IO',
            'Chemo + PD-L1': 'Chemo + IO',
            'CTLA-4 + PD-1 combo': 'IO combo (CTLA-4 + PD-1)',
            'IO-contraindicated': 'Contraindicated'
        }
        if df['io_type'].notna().any():
            df['io_group'] = df['io_type'].map(map_to_group)
        else:
            df['io_group'] = np.nan
    else:
        df['io_group'] = np.nan

    # Yaş metinse sayıya map et
    if df['age'].dtype == 'O':
        age_map = {'Young': 30, 'Middle-Aged': 60, 'Senior': 75, 'Elderly': 85}
        df['age'] = df['age'].map(age_map).fillna(df['age']).astype(float)

    target_resp = "response"
    features = [
        'pd_l1', 'tmb', 'age', 'msi', 'metastasis',
        'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days'
    ]
    X = df[features].copy()
    y = df[target_resp].astype(int).copy()

    numeric_features = ['tmb', 'pd_l1', 'age', 'ecog_score', 'treatment_start_days']
    categorical_features = ['msi', 'smoking_status', 'cancer_stage']
    preprocessor = make_preprocessor(numeric_features, categorical_features)

    X_processed = preprocessor.fit_transform(X)
    X_processed = ensure_float32(X_processed)
    feature_names = preprocessor.get_feature_names_out()

    # IO_GROUP varsa string etiketleri döndür
    has_group = df['io_group'].notna().any()
    y_group_str = df['io_group'].astype(str) if has_group else None

    return df, X, y, y_group_str, preprocessor, X_processed, feature_names, features

# ============================
# ANA AKIŞ
# ============================
if __name__ == "__main__":
    # Colab: CSV_PATH = "final_dataset.csv"
    CSV_PATH = "data/final_dataset.csv"

    df, X, y, y_group_str, preprocessor, X_processed, feature_names, features = load_and_prepare(CSV_PATH)

    # ---- RESPONSE ----
    print("👉 Class balance (response=0/1):", y.value_counts().to_dict())
    resp = train_and_evaluate_response(X_processed, y, random_state=42)
    cal_model_resp   = resp["cal_model"]
    X_train_resp     = resp["X_train"]
    best_thresh      = resp["best_thresh"]

    print(f"\n🔍 RESPONSE — Optimum Eşik (F1): {best_thresh:.4f}")
    print("\n📊 RESPONSE Classification Report:\n", resp["report"])
    print("✅ RESPONSE Accuracy:", resp["acc"])
    print("📈 RESPONSE ROC AUC:", resp["auc"])

    # ---- IO_GROUP (opsiyonel) ----
    if y_group_str is not None and y_group_str.notna().any():
        io_out = train_and_evaluate_io_group(X_processed, y_group_str, random_state=42)
        cal_model_group = io_out["cal_model"]
        io_classes      = io_out["classes"]
        print("\n📊 IO_GROUP Classification Report:\n", io_out["report"])
        print("✅ IO_GROUP Accuracy:", io_out["acc"])
        print("🎯 IO_GROUP Macro-F1:", io_out["f1_macro"])
    else:
        cal_model_group = None
        io_classes = None
        print("\nℹ️ 'io_type' bulunamadı → IO_GROUP tahmini atlandı.")

    # ---- SHAP (response) ----
    X_train_dense = to_dense(X_train_resp)
    explainer = build_calibrated_permutation_explainer(
        cal_model=cal_model_resp,
        X_train_dense=X_train_dense,
        feature_names=feature_names
    )

    # ---- Tek hasta örneği ----
    idx = 64  # istersen değiştir; ör: df.index[df['response']==1][0]
    row, sample_df = select_patient_row(df, features, iloc_index=idx)
    sample_X = preprocessor.transform(sample_df)
    sample_X_dense = to_dense(sample_X)
    hasta_df = pd.DataFrame(sample_X_dense, columns=feature_names)

    # Fuzzy + response
    fuzzy = compute_fuzzy_success_factor(row)
    success_factor = fuzzy["success_factor"]
    proba_model_raw = float(cal_model_resp.predict_proba(sample_X_dense)[0, 1])
    proba_model = float(np.clip(proba_model_raw, 1e-6, 1 - 1e-6))
    proba_final = 1 - (1 - proba_model) ** success_factor
    proba_final = float(min(proba_final, 0.65))  # istersen kaldır

    # IO_GROUP tek hasta (varsa)
    if cal_model_group is not None:
        proba_groups = cal_model_group.predict_proba(sample_X_dense)[0]
        top_idx = np.argsort(proba_groups)[::-1]
        top3 = [(io_classes[i], float(proba_groups[i])) for i in top_idx[:3]]
        pred_group = io_classes[top_idx[0]]
    else:
        top3 = []; pred_group = None

    # Çıktılar
    print("\n──── Tek Hasta Özeti ────")
    print(f"🔹 Kalibre Edilmiş Model Olasılığı (raw): {proba_model_raw:.6f}")
    print(f"🔹 Kalibre Edilmiş Model Olasılığı (clipped): {proba_model:.6f}")
    print(f"🔹 Fuzzy Başarı Katsayısı: {success_factor:.2f}")
    print(f"✅ Bu hasta için tedavi başarılı olma olasılığı (fuzzy harman): %{proba_final*100:.2f}")

    if pred_group is not None:
        print(f"\n💊 Önerilen IO Grup (tahmin): {pred_group}")
        print("   Top-3 grup ve olasılıkları:")
        for name, p in top3:
            print(f"   • {name}: %{p*100:.2f}")

    print(
        "\nÖzellikler -> "
        f"PD-L1:{row['pd_l1']}, TMB:{row['tmb']}, ECOG:{row['ecog_score']}, "
        f"Gecikme:{row['treatment_start_days']}, Yaş:{row['age']}, "
        f"Sigara:{safe_lower(row['smoking_status'])}, Evre:{row['cancer_stage']}"
    )

    # SHAP waterfall
    try:
        shap_values = explainer(hasta_df)
        import matplotlib.pyplot as plt
        shap.plots.waterfall(shap_values[0], show=False)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print("SHAP waterfall oluşturulurken bir uyarı/hata oluştu:", str(e))